# ACDC Diastole → 3D LV Reconstruction
## Real-life prediction using ACDC segmentation + trained INR Occupancy Network

**Pipeline:**
1. Load ACDC diastole GT segmentation (NIfTI)
2. Extract 2D contour points per SAX slice → world-space XYZ
3. **Preview stacked ACDC slices** before prediction
4. Normalise contours (centroid + scale) to match training distribution
5. Run the trained `OccupancyNetwork` → marching-cubes → 3D endo/epi meshes
6. Visualise the reconstructed LV

> ⚠️ **Diastole only** — this model was trained on end-diastolic frames. 
> `patient001_frame01` is the ED frame in ACDC convention.

In [ ]:
%pip install -q nibabel scikit-image trimesh scipy

In [ ]:
import os, warnings, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

import nibabel as nib
from skimage.measure import find_contours
from skimage.morphology import binary_erosion, disk
from scipy.ndimage import label as nd_label
import trimesh
from skimage.measure import marching_cubes

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Config (must match training exactly) ─────────────────────
CFG = dict(
    latent_dim     = 256,
    fourier_L      = 6,
    decoder_hidden = 512,
    decoder_layers = 8,
    skip_layer     = 4,
    grid_res       = 64,
    iso_thresh     = 0.5,
)

# ── Paths ─────────────────────────────────────────────────────
CKPT_PATH = '/kaggle/input/models/andrefce/inr-final/pytorch/inr_v2/1/inr_final(1).pt'    # trained checkpoint
ACDC_GT   = ('/kaggle/input/datasets/andrefce/realmri/training/patient001/patient001_frame12_gt.nii/DCM03-OH-AL_V2_12.nii')
# ── ACDC label legend ─────────────────────────────────────────
# 0 = Background
# 1 = RV  (right ventricle)
# 2 = Myocardium  (the muscle wall — between endo and epi)
# 3 = LV  (left-ventricle blood pool = endocardium boundary)
LBL_BG, LBL_RV, LBL_MYO, LBL_LV = 0, 1, 2, 3

# Points sampled per contour ring per slice
N_PTS_PER_RING = 60

print('Config OK')
print(f'  Checkpoint : {CKPT_PATH}')
print(f'  ACDC GT    : {ACDC_GT}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Model architecture  (must be identical to training)
# ══════════════════════════════════════════════════════════════

class FourierPE(nn.Module):
    def __init__(self, L=6):
        super().__init__()
        self.L = L
        self.register_buffer('freqs', 2.0 ** torch.arange(L).float() * np.pi)
        self.out_dim = 3 + 6 * L

    def forward(self, xyz):
        angles = xyz.unsqueeze(-1) * self.freqs       # (..., 3, L)
        enc    = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        enc    = enc.reshape(*xyz.shape[:-1], 6 * self.L)
        return torch.cat([xyz, enc], dim=-1)


class PointNetEncoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Linear(4, 64),  nn.LayerNorm(64),  nn.ReLU())
        self.conv2 = nn.Sequential(nn.Linear(64, 128), nn.LayerNorm(128), nn.ReLU())
        self.conv3 = nn.Sequential(nn.Linear(128, 256), nn.LayerNorm(256), nn.ReLU())
        self.tissue_proj = nn.Sequential(nn.Linear(256, 128), nn.ReLU())
        self.agg = nn.Sequential(
            nn.Linear(512, 512), nn.LayerNorm(512), nn.ReLU(),
            nn.Linear(512, latent_dim))

    def forward(self, x, mask):
        B, N, _ = x.shape
        h = self.conv1(x); h = self.conv2(h); h = self.conv3(h)
        neg_inf   = torch.full_like(h, float('-inf'))
        h_masked  = torch.where(mask.unsqueeze(-1), h, neg_inf)
        g_all     = h_masked.max(dim=1).values
        tissue    = x[:, :, 3]
        hp        = self.tissue_proj(h)

        def tissue_pool(label):
            t_mask = (tissue == label) & mask
            t_neg  = torch.full_like(hp, float('-inf'))
            t_h    = torch.where(t_mask.unsqueeze(-1), hp, t_neg)
            pooled = t_h.max(dim=1).values
            has    = t_mask.any(dim=1, keepdim=True).float()
            return pooled * has

        g_endo = tissue_pool(0.0)
        g_epi  = tissue_pool(1.0)
        return self.agg(torch.cat([g_all, g_endo, g_epi], dim=-1))


# Inference INRDecoder — same 4-block structure, dropout disabled
class INRDecoder(nn.Module):
    def __init__(self, latent_dim=256, fourier_L=6, hidden=512, n_layers=8, skip_layer=4, dropout=0.0):
        super().__init__()
        self.skip_layer = skip_layer
        pe_dim = 3 + 6 * fourier_L
        in_dim = latent_dim + pe_dim
        layers = []
        cur_dim = in_dim
        for i in range(n_layers):
            if i == skip_layer:
                cur_dim += in_dim
            layers += [nn.Linear(cur_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(dropout)]
            cur_dim = hidden
        self.layers    = nn.ModuleList(layers)
        self.head_endo = nn.Linear(hidden, 1)
        self.head_epi  = nn.Linear(hidden, 1)
        self._in_dim   = in_dim

    def forward(self, z, pe_xyz):
        B, Q, _ = pe_xyz.shape
        z_exp = z.unsqueeze(1).expand(-1, Q, -1)
        inp   = torch.cat([z_exp, pe_xyz], dim=-1)
        h = inp; step = 0
        for j in range(0, len(self.layers), 4):
            if step == self.skip_layer:
                h = torch.cat([h, inp], dim=-1)
            h = self.layers[j](h); h = self.layers[j+1](h)
            h = self.layers[j+2](h); h = self.layers[j+3](h)
            step += 1
        return self.head_endo(h).squeeze(-1), self.head_epi(h).squeeze(-1)


class OccupancyNetwork(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder = PointNetEncoder(latent_dim=cfg['latent_dim'])
        self.fourier = FourierPE(L=cfg['fourier_L'])
        self.decoder = INRDecoder(
            latent_dim=cfg['latent_dim'], fourier_L=cfg['fourier_L'],
            hidden=cfg['decoder_hidden'], n_layers=cfg['decoder_layers'],
            skip_layer=cfg['skip_layer'])

    def encode(self, contour, mask):
        return self.encoder(contour, mask)

    def decode(self, z, query_xyz):
        pe = self.fourier(query_xyz)
        return self.decoder(z, pe)

    def forward(self, contour, mask, query_xyz):
        z = self.encode(contour, mask)
        return self.decode(z, query_xyz)


model = OccupancyNetwork(CFG).to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── Load trained checkpoint ───────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'✅ Checkpoint loaded')
print(f'   Epoch    : {ckpt.get("epoch", "?")}')
print(f'   Val loss : {ckpt.get("val_loss", float("nan")):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Load ACDC diastole GT segmentation
# ══════════════════════════════════════════════════════════════

nii  = nib.load(ACDC_GT)
data = nii.get_fdata().astype(np.int32)   # (H, W, n_slices)  or  (H, W, 1, n_slices)

# Some ACDC NIfTI files have an extra singleton time dimension
if data.ndim == 4:
    data = data[:, :, :, 0]

affine = nii.affine    # 4x4 voxel-to-world (mm) matrix
H, W, S = data.shape

print(f'NIfTI shape : {data.shape}   dtype: {data.dtype}')
print(f'Affine:\n{affine.round(3)}')
print(f'Unique labels : {np.unique(data)}')
print(f'Slices with LV  label (3) : {(data == LBL_LV).any(axis=(0,1)).sum()}')
print(f'Slices with Myo label (2) : {(data == LBL_MYO).any(axis=(0,1)).sum()}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# PREVIEW — stacked ACDC diastole GT slices
# Slices are shown in ascending index order: low index = BASE, high index = APEX
# ══════════════════════════════════════════════════════════════

lv_slices = sorted([s for s in range(S)
                    if (data[:,:,s] == LBL_LV).any() or (data[:,:,s] == LBL_MYO).any()])

print(f'Slices with LV/Myo annotation (base → apex): {lv_slices}')
n_show = len(lv_slices)

CMAP = {0: [0.10, 0.10, 0.10],
        1: [0.20, 0.60, 0.90],
        2: [0.95, 0.55, 0.15],
        3: [0.20, 0.80, 0.35]}

def seg_to_rgb(seg2d):
    H2, W2 = seg2d.shape
    rgb = np.zeros((H2, W2, 3))
    for lbl, colour in CMAP.items():
        mask = seg2d == lbl
        for c in range(3):
            rgb[:,:,c] += mask * colour[c]
    return rgb

ncols = min(n_show, 8)
nrows = (n_show + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols,
                         figsize=(ncols * 2.8, nrows * 3.0),
                         facecolor='#0d0d0d')
axes = np.array(axes).reshape(nrows, ncols)

for i, s in enumerate(lv_slices):
    ax = axes[i // ncols, i % ncols]
    ax.imshow(seg_to_rgb(data[:,:,s]), origin='lower', interpolation='nearest')

    if i == 0:
        extra = '  ← BASE'
    elif i == len(lv_slices) - 1:
        extra = '  ← APEX'
    else:
        extra = ''
    ax.set_title(f'Slice {s}{extra}', color='white', fontsize=8, pad=3)
    ax.axis('off')

    if i == 0:
        ax.set_frame_on(True)
        for spine in ax.spines.values():
            spine.set_edgecolor('#00ffff'); spine.set_linewidth(2.5)
    elif i == len(lv_slices) - 1:
        ax.set_frame_on(True)
        for spine in ax.spines.values():
            spine.set_edgecolor('#ff44ff'); spine.set_linewidth(2.5)

for j in range(n_show, nrows * ncols):
    axes[j // ncols, j % ncols].axis('off')

patches = [
    mpatches.Patch(color=CMAP[1], label='RV (1)'),
    mpatches.Patch(color=CMAP[2], label='Myocardium (2)'),
    mpatches.Patch(color=CMAP[3], label='LV (3)'),
    mpatches.Patch(color='#00ffff', label='Base (lowest slice index)'),
    mpatches.Patch(color='#ff44ff', label='Apex (highest slice index)'),
]
fig.legend(handles=patches, loc='lower center', ncol=5,
           fontsize=9, framealpha=0.0,
           labelcolor='white', facecolor='#0d0d0d',
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle(
    'ACDC Patient 001 — Diastole (frame 01) — GT Segmentation\n'
    'SAX slices in order: BASE (cyan border) → APEX (magenta border)',
    color='white', fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig('/kaggle/working/acdc_slices_preview.png',
            dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('✅ Preview saved → /kaggle/working/acdc_slices_preview.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Extract 2D contour points per slice → 3D world-space XYZ
# ══════════════════════════════════════════════════════════════

DZ = float(nii.header.get_zooms()[2])   # 10.0 mm
print(f'Using slice thickness dz = {DZ} mm')

def voxel_to_world(rows, cols, slice_idx, affine):
    """x/y from affine (has correct pixel spacing), z from header dz."""
    vox   = np.column_stack([cols, rows,
                             np.zeros(len(rows)),   # z=0, we set it manually
                             np.ones(len(rows))])
    world = (affine @ vox.T).T
    world[:, 2] = slice_idx * DZ          # ← correct z in mm
    return world[:, :3]

def contour_area(pts):
    """Shoelace formula — returns enclosed area of a contour ring."""
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def sample_contour(pts2d, n=N_PTS_PER_RING):
    if len(pts2d) < 3:
        return pts2d
    # np.diff gives N-1 gaps → cumdist is length N, same as pts2d ✓
    diffs   = np.diff(pts2d, axis=0)
    dists   = np.sqrt((diffs ** 2).sum(axis=1))
    total   = dists.sum()
    if total < 1e-6:
        return pts2d
    cumdist = np.concatenate([[0], np.cumsum(dists)])   # length N
    t_new   = np.linspace(0, total, n, endpoint=False)
    rows_r  = np.interp(t_new, cumdist, pts2d[:, 0])
    cols_r  = np.interp(t_new, cumdist, pts2d[:, 1])
    return np.column_stack([rows_r, cols_r])

all_contour_pts = []

for s in lv_slices:
    seg = data[:, :, s]

    lv_mask = (seg == LBL_LV).astype(np.uint8)
    if lv_mask.sum() > 10:
        contours_endo = find_contours(lv_mask, 0.5)
        if contours_endo:
            ring  = max(contours_endo, key=contour_area)
            ring  = sample_contour(ring, N_PTS_PER_RING)
            world = voxel_to_world(ring[:, 0], ring[:, 1], s, affine)
            all_contour_pts.append(np.column_stack([world, np.zeros(len(world))]))

    lv_myo_mask = ((seg == LBL_MYO) | (seg == LBL_LV)).astype(np.uint8)
    if lv_myo_mask.sum() > 10:
        contours_epi = find_contours(lv_myo_mask, 0.5)
        if contours_epi:
            ring  = max(contours_epi, key=contour_area)  # area → always outer ring
            ring  = sample_contour(ring, N_PTS_PER_RING)
            world = voxel_to_world(ring[:, 0], ring[:, 1], s, affine)
            all_contour_pts.append(np.column_stack([world, np.ones(len(world))]))

contour_raw = np.vstack(all_contour_pts).astype(np.float32)
print(f'Total contour points : {len(contour_raw)}')
print(f'  Endo points (0)    : {(contour_raw[:,3]==0).sum()}')
print(f'  Epi  points (1)    : {(contour_raw[:,3]==1).sum()}')
print(f'XYZ range (mm):')
print(f'  x : [{contour_raw[:,0].min():.1f}, {contour_raw[:,0].max():.1f}]')
print(f'  y : [{contour_raw[:,1].min():.1f}, {contour_raw[:,1].max():.1f}]')
print(f'  z : [{contour_raw[:,2].min():.1f}, {contour_raw[:,2].max():.1f}]')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Diagnose affine z-spacing, then normalise
# ══════════════════════════════════════════════════════════════

xyz_raw  = contour_raw[:, :3]
tissue   = contour_raw[:, 3]

centroid = xyz_raw.mean(0)
xyz_c    = xyz_raw - centroid
scale    = np.linalg.norm(xyz_c[:, :2], axis=1).mean()   # match training
xyz_n    = (xyz_c / scale).astype(np.float32)

print(f'scale: {scale:.2f} mm  (training was ~25.4 mm)')
print(f'x: [{xyz_n[:,0].min():.3f}, {xyz_n[:,0].max():.3f}]')
print(f'y: [{xyz_n[:,1].min():.3f}, {xyz_n[:,1].max():.3f}]')
print(f'z: [{xyz_n[:,2].min():.3f}, {xyz_n[:,2].max():.3f}]  ← target ~±1.9')
print(f'max abs: {np.abs(xyz_n).max():.3f}               ← training was 1.962')

contour_norm = np.column_stack([xyz_n, tissue]).astype(np.float32)

# ── Step 4: flip z to match training convention ───────────────
# Training data: base = low z_n (negative), apex = high z_n (positive).
# ACDC NIfTI: slice 0 = base = lowest z_world → after centering that's
# the most negative z_n → apex ends up at positive z_n = top of the model.
# If the reconstruction still looks inverted after running, toggle this:
FLIP_Z = True
if FLIP_Z:
    xyz_n[:, 2] = -xyz_n[:, 2]
    print('z flipped (FLIP_Z=True) — set to False if shape looks inverted')

contour_norm = np.column_stack([xyz_n, tissue]).astype(np.float32)

print(f'\nCentroid  : {centroid.round(2)}')
print(f'xyz_n range: [{xyz_n.min():.3f}, {xyz_n.max():.3f}]')
print(f'  z_n range: [{xyz_n[:,2].min():.3f}, {xyz_n[:,2].max():.3f}]  '
      f'(should span ≈ ±0.3–0.8 for a full LV stack)')

In [ ]:
# ── Quick 3D scatter of the extracted contour ────────────────
fig = plt.figure(figsize=(9, 7), facecolor='#0d0d0d')
ax  = fig.add_subplot(111, projection='3d', facecolor='#0d0d0d')

endo_pts = xyz_n[tissue == 0]
epi_pts  = xyz_n[tissue == 1]

ax.scatter(*endo_pts.T, c='#27ae60', s=12, alpha=0.8, label='Endo (tissue=0)')
ax.scatter(*epi_pts.T,  c='#e74c3c', s=12, alpha=0.8, label='Epi  (tissue=1)')

ax.set_xlabel('x (norm)', color='white'); ax.set_ylabel('y (norm)', color='white')
ax.set_zlabel('z (norm)', color='white')
ax.tick_params(colors='white')
ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
ax.legend(facecolor='#1a1a1a', labelcolor='white', edgecolor='white')
ax.set_title('Normalised SAX Contour Points\n(input to INR encoder)',
             color='white', fontsize=12, fontweight='bold')
ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig('/kaggle/working/acdc_contour_3d.png',
            dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

In [ ]:
def predict_mesh(model, contour_xyz, tissue_labels, cfg,
                 grid_res=None, iso_thresh=None, batch_query=65536):
    if grid_res   is None: grid_res   = cfg['grid_res']
    if iso_thresh is None: iso_thresh = cfg['iso_thresh']

    model.eval()
    cont   = np.column_stack([contour_xyz, tissue_labels]).astype(np.float32)
    cont_t = torch.from_numpy(cont).unsqueeze(0).to(DEVICE)
    mask_t = torch.ones(1, len(cont), dtype=torch.bool).to(DEVICE)

    with torch.no_grad():
        z = model.encode(cont_t, mask_t)

    pad = 0.15
    lo  = contour_xyz.min(0) - pad
    hi  = contour_xyz.max(0) + pad
    ext = hi - lo
    vox = ext.max() / (grid_res - 1)
    nx, ny, nz = [max(4, int(np.ceil(e / vox)) + 1) for e in ext]
    xs = np.linspace(lo[0], hi[0], nx)
    ys = np.linspace(lo[1], hi[1], ny)
    zs = np.linspace(lo[2], hi[2], nz)
    gx, gy, gz = np.meshgrid(xs, ys, zs, indexing='ij')
    grid_pts   = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=-1).astype(np.float32)
    print(f'  Grid: {nx}×{ny}×{nz}  ({len(grid_pts):,} pts)  voxel≈{vox:.4f}  extents x={ext[0]:.2f} y={ext[1]:.2f} z={ext[2]:.2f}')

    endo_logits = np.zeros(len(grid_pts), np.float32)
    epi_logits  = np.zeros(len(grid_pts), np.float32)

    for start in range(0, len(grid_pts), batch_query):
        pts = torch.from_numpy(grid_pts[start:start+batch_query]).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            el, pl = model.decode(z, pts)
        endo_logits[start:start+batch_query] = el[0].cpu().numpy()
        epi_logits[ start:start+batch_query] = pl[0].cpu().numpy()

    occ_endo = torch.sigmoid(torch.from_numpy(endo_logits)).numpy().reshape(nx, ny, nz)
    occ_epi  = torch.sigmoid(torch.from_numpy(epi_logits )).numpy().reshape(nx, ny, nz)

    voxel_size = (hi - lo) / (np.array([nx, ny, nz]) - 1)

    def vol_to_mesh(occ):
        if occ.max() < iso_thresh:
            print('  ⚠ No surface above threshold — check normalisation / model.')
            return trimesh.Trimesh()
        try:
            verts, faces, _, _ = marching_cubes(occ, level=iso_thresh, spacing=voxel_size)
            verts = verts + lo
            return trimesh.Trimesh(vertices=verts, faces=faces, process=True)
        except Exception as e:
            print(f'  Marching cubes failed: {e}')
            return trimesh.Trimesh()

    return vol_to_mesh(occ_endo), vol_to_mesh(occ_epi), occ_endo, occ_epi


print('predict_mesh ready')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Run the model on ACDC diastole contour
# ══════════════════════════════════════════════════════════════

print('Running INR occupancy network on ACDC patient001 diastole…')
endo_mesh, epi_mesh, occ_endo, occ_epi = predict_mesh(
    model, xyz_n, tissue, CFG, grid_res=CFG['grid_res'])

print(f'\n✅ Prediction done')
print(f'   Endo mesh : {len(endo_mesh.vertices):,} vertices  |  {len(endo_mesh.faces):,} faces')
print(f'   Epi  mesh : {len(epi_mesh.vertices):,} vertices  |  {len(epi_mesh.faces):,} faces')
print(f'   Endo occ range : [{occ_endo.min():.3f}, {occ_endo.max():.3f}]')
print(f'   Epi  occ range : [{occ_epi.min():.3f},  {occ_epi.max():.3f}]')

In [ ]:
def draw_mesh(ax, mesh, color, alpha, max_faces=5000):
    if len(mesh.faces) == 0: return
    sel = mesh.faces
    if len(sel) > max_faces:
        sel = sel[np.random.choice(len(sel), max_faces, replace=False)]
    ax.add_collection3d(Poly3DCollection(
        mesh.vertices[sel], alpha=alpha,
        facecolor=color, edgecolor='none'))

def _lim(ax, pts):
    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) == 0: return
    vmin = pts.min(0);  vmax = pts.max(0)
    ext  = vmax - vmin
    ctr  = (vmin + vmax) / 2
    pad  = ext.max() * 0.08
    ax.set_xlim(ctr[0] - ext[0]/2 - pad, ctr[0] + ext[0]/2 + pad)
    ax.set_ylim(ctr[1] - ext[1]/2 - pad, ctr[1] + ext[1]/2 + pad)
    ax.set_zlim(ctr[2] - ext[2]/2 - pad, ctr[2] + ext[2]/2 + pad)
    aspect = ext + 2 * pad
    aspect = np.maximum(aspect, aspect.max() * 0.15)
    ax.set_box_aspect(aspect)

plt.style.use('dark_background')
fig = plt.figure(figsize=(28, 12), facecolor='#0d0d0d')
fig.suptitle(
    'ACDC Patient 047 — Diastole — 3D LV Reconstruction\n'
    'INR Occupancy Network (PointNet encoder + Fourier INR decoder)',
    color='white', fontsize=13, fontweight='bold', y=0.98)

all_pts = np.vstack([
    endo_mesh.vertices if len(endo_mesh.vertices) > 0 else np.zeros((1,3)),
    epi_mesh.vertices  if len(epi_mesh.vertices)  > 0 else np.zeros((1,3)),
])

gs = fig.add_gridspec(1, 5, wspace=0.05)

panels = [
    (0, 'Input SAX Contours',     20, 45),
    (1, 'Predicted Endo',         20, 45),
    (2, 'Predicted Endo + Epi',   20, 45),
    (3, 'Mesh + SAX Contours',    20, 45),
    (4, 'Top View (short-axis)',  88,  0),
]

for col, title, elev, azim in panels:
    ax = fig.add_subplot(gs[col], projection='3d', facecolor='#0d0d0d')
    ax.set_title(title, color='white', fontsize=10, fontweight='bold', pad=8)
    ax.set_axis_off()
    ax.view_init(elev=elev, azim=azim)

    if col == 0:
        ax.scatter(*xyz_n[tissue==0].T, c='#27ae60', s=12, alpha=0.9, label='Endo')
        ax.scatter(*xyz_n[tissue==1].T, c='#e74c3c', s=12, alpha=0.9, label='Epi')
        ax.legend(fontsize=7, facecolor='#1a1a1a', labelcolor='white', loc='upper left')
    elif col == 1:
        draw_mesh(ax, endo_mesh, '#27ae60', 0.85)
    elif col == 2:
        draw_mesh(ax, epi_mesh,  '#e74c3c', 0.35)
        draw_mesh(ax, endo_mesh, '#27ae60', 0.85)
    elif col == 3:
        draw_mesh(ax, epi_mesh,  '#e74c3c', 0.30)
        draw_mesh(ax, endo_mesh, '#27ae60', 0.75)
        ax.scatter(*xyz_n[tissue==0].T, c='#7fbf7f', s=4, alpha=0.60, depthshade=True)
        ax.scatter(*xyz_n[tissue==1].T, c='#bf7f7f', s=4, alpha=0.60, depthshade=True)
    elif col == 4:
        draw_mesh(ax, epi_mesh,  '#e74c3c', 0.35)
        draw_mesh(ax, endo_mesh, '#27ae60', 0.85)

    _lim(ax, all_pts)

plt.savefig('/kaggle/working/acdc_lv_reconstruction.png',
            dpi=300, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
plt.style.use('default')
print('✅ Saved → /kaggle/working/acdc_lv_reconstruction.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Occupancy field cross-sections (sanity check)
# ══════════════════════════════════════════════════════════════

mid = CFG['grid_res'] // 2

fig, axes = plt.subplots(2, 3, figsize=(14, 8), facecolor='#0d0d0d')
titles = ['Axial (z=mid)', 'Coronal (y=mid)', 'Sagittal (x=mid)']
slices_e = [occ_endo[:,:,mid], occ_endo[:,mid,:], occ_endo[mid,:,:]]
slices_p = [occ_epi[:,:,mid],  occ_epi[:,mid,:],  occ_epi[mid,:,:]]

for col, (title, se, sp) in enumerate(zip(titles, slices_e, slices_p)):
    for row, (occ, label, cmap) in enumerate([(se, 'Endo', 'Greens'),
                                               (sp, 'Epi',  'Reds')]):
        ax = axes[row, col]
        ax.imshow(occ.T, origin='lower', cmap=cmap, vmin=0, vmax=1)
        ax.contour(occ.T, levels=[0.5], colors='white', linewidths=1.2)
        ax.set_title(f'{label}  —  {title}', color='white', fontsize=9)
        ax.axis('off')

fig.suptitle('Occupancy Field Cross-Sections (white contour = 0.5 threshold)',
             color='white', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/acdc_occ_crosssections.png',
            dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
plt.style.use('default')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Wall thickness estimate (endo → epi nearest-neighbour)
# ══════════════════════════════════════════════════════════════

if len(endo_mesh.vertices) > 0 and len(epi_mesh.vertices) > 0:
    from sklearn.neighbors import KDTree

    # For each endo vertex, find nearest epi vertex
    epi_tree  = KDTree(epi_mesh.vertices)
    dists, _  = epi_tree.query(endo_mesh.vertices, k=1)
    thickness = dists[:, 0] * scale

    
    print(f'Wall thickness (normalised units → mm, scale={scale:.1f} mm):')
    print(f'  Mean  : {thickness.mean():.2f} mm')
    print(f'  Median: {np.median(thickness):.2f} mm')
    print(f'  Std   : {thickness.std():.2f} mm')
    print(f'  Range : [{thickness.min():.2f}, {thickness.max():.2f}] mm')
    print()
    print('  Typical healthy LV wall thickness (ED): 6–12 mm')
else:
    print('⚠ One or both meshes are empty — cannot compute wall thickness.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Extract dense point cloud from ACDC segmentation (all voxels)
# ══════════════════════════════════════════════════════════════

print('Extracting dense point cloud from ACDC segmentation...')

dense_pts_raw = []
for s in lv_slices:
    seg = data[:, :, s]
    lv_mask  = (seg == LBL_LV).astype(bool)
    myo_mask = (seg == LBL_MYO).astype(bool)
    lv_idx  = np.where(lv_mask)
    myo_idx = np.where(myo_mask)
    if len(lv_idx[0]) > 0:
        world = voxel_to_world(lv_idx[0],  lv_idx[1],  s, affine)
        dense_pts_raw.append(np.column_stack([world, np.zeros(len(world))]))
    if len(myo_idx[0]) > 0:
        world = voxel_to_world(myo_idx[0], myo_idx[1], s, affine)
        dense_pts_raw.append(np.column_stack([world, np.ones(len(world))]))

if dense_pts_raw:
    dense_pts_all  = np.vstack(dense_pts_raw).astype(np.float32)
    dense_xyz      = dense_pts_all[:, :3]
    dense_tissue   = dense_pts_all[:, 3]
    dense_centroid = dense_xyz.mean(0)
    dense_xyz_c    = dense_xyz - dense_centroid
    dense_scale    = np.linalg.norm(dense_xyz_c[:, :2], axis=1).mean()
    dense_xyz_n    = (dense_xyz_c / dense_scale).astype(np.float32)
    if FLIP_Z:
        dense_xyz_n[:, 2] = -dense_xyz_n[:, 2]
    print(f'Dense point cloud : {len(dense_xyz_n):,} points  scale={dense_scale:.2f} mm')
    print(f'  LV: {(dense_tissue==0).sum():,}  Myo: {(dense_tissue==1).sum():,}')
else:
    print('⚠ No dense points extracted')
    dense_xyz_n  = xyz_n
    dense_tissue = tissue

# ══════════════════════════════════════════════════════════════
# Reconstruction methods
# ══════════════════════════════════════════════════════════════

def safe_mesh(func, *args, **kwargs):
    try:
        m = func(*args, **kwargs)
        return m if (isinstance(m, trimesh.Trimesh) and len(m.vertices) > 0) else trimesh.Trimesh()
    except Exception as e:
        print(f'    ⚠ {func.__name__}: {str(e)[:80]}')
        return trimesh.Trimesh()


# ── 1. Convex Hull ─────────────────────────────────────────────
# Trivial upper-bound baseline — fastest possible closed surface.
def mesh_via_convex_hull(pts):
    return trimesh.convex.convex_hull(pts)


# ── 2. Alpha Shape (boundary-only) ─────────────────────────────
# Concave hull: keeps only tetrahedra faces with circumradius < 1/α
# that appear exactly once → outer boundary surface.
def mesh_via_alpha_shape(pts, alpha=0.3):
    from scipy.spatial import Delaunay
    from collections import defaultdict
    tri   = Delaunay(pts)
    face_count = defaultdict(int)
    for s in tri.simplices:
        p = pts[s]
        # circumradius of tetrahedron
        a  = np.linalg.norm(p[1]-p[0]); b = np.linalg.norm(p[2]-p[0])
        c  = np.linalg.norm(p[2]-p[1]); d = np.linalg.norm(p[3]-p[0])
        e  = np.linalg.norm(p[3]-p[1]); f = np.linalg.norm(p[3]-p[2])
        vol = abs(np.dot(p[1]-p[0], np.cross(p[2]-p[0], p[3]-p[0]))) / 6.0
        if vol < 1e-12: continue
        # Cayley-Menger circumradius
        R = (a*f * b*e * c*d) ** (1/3) / (6 * vol + 1e-12)
        if R < 1.0 / alpha:
            for face in [(0,1,2),(0,1,3),(0,2,3),(1,2,3)]:
                key = tuple(sorted([s[i] for i in face]))
                face_count[key] += 1
    boundary = [list(k) for k, v in face_count.items() if v == 1]
    if not boundary:
        return trimesh.Trimesh()
    return trimesh.Trimesh(vertices=pts, faces=np.array(boundary), process=True)


# ── 3. Gaussian Implicit Surface ───────────────────────────────
# KDE on a 3D grid → marching cubes at 30% of peak density.
# Represents each voxel as a Gaussian blob; the iso-surface is the
# smooth hull enclosing the bulk of the point mass.
def mesh_via_gaussian(pts, sigma=0.08, grid_res=48):
    from scipy.ndimage import gaussian_filter
    from scipy.spatial import cKDTree
    lo = pts.min(0) - 0.1;  hi = pts.max(0) + 0.1
    ext = hi - lo
    # Anisotropic grid so voxels stay cubic
    vox  = ext.max() / (grid_res - 1)
    nx, ny, nz = [max(4, int(np.ceil(e / vox)) + 1) for e in ext]
    xs = np.linspace(lo[0], hi[0], nx)
    ys = np.linspace(lo[1], hi[1], ny)
    zs = np.linspace(lo[2], hi[2], nz)
    xx, yy, zz = np.meshgrid(xs, ys, zs, indexing='ij')
    gpts = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], axis=1)
    dists, _ = cKDTree(pts).query(gpts, k=1)
    density  = np.exp(-dists**2 / (2*sigma**2)).reshape(nx, ny, nz)
    density  = gaussian_filter(density, sigma=1.0)
    spacing  = (hi - lo) / (np.array([nx, ny, nz]) - 1)
    verts, faces, _, _ = marching_cubes(density, level=density.max()*0.30,
                                        spacing=spacing)
    return trimesh.Trimesh(vertices=verts + lo, faces=faces, process=True)


# ── 4. Slice Loft ──────────────────────────────────────────────
# Re-samples the contour ring at each SAX slice and linearly
# interpolates (lofts) between consecutive rings — the classical
# clinical approach before 3D reconstruction became standard.
def mesh_via_slice_loft(slices_list, n_ring=40):
    rings_endo, rings_epi = [], []
    for s in slices_list:
        seg = data[:, :, s]
        for mask_fn, ring_list in [
            (lambda g: (g == LBL_LV).astype(np.uint8),               rings_endo),
            (lambda g: ((g == LBL_MYO)|(g == LBL_LV)).astype(np.uint8), rings_epi),
        ]:
            m = mask_fn(seg)
            if m.sum() < 10: continue
            ctrs = find_contours(m, 0.5)
            if not ctrs: continue
            ring  = max(ctrs, key=contour_area)
            ring  = sample_contour(ring, n_ring)
            world = voxel_to_world(ring[:,0], ring[:,1], s, affine)
            xyz_c = (world - dense_centroid) / dense_scale
            if FLIP_Z: xyz_c[:, 2] = -xyz_c[:, 2]
            ring_list.append(xyz_c.astype(np.float32))

    def loft(rings):
        if len(rings) < 2: return trimesh.Trimesh()
        all_v, all_f, off = [], [], 0
        for r1, r2 in zip(rings[:-1], rings[1:]):
            n = max(len(r1), len(r2))
            r1 = r1[np.linspace(0, len(r1)-1, n, dtype=int)]
            r2 = r2[np.linspace(0, len(r2)-1, n, dtype=int)]
            v1, v2 = off, off + n
            all_v.extend(r1); all_v.extend(r2); off += 2*n
            for j in range(n):
                jn = (j+1) % n
                all_f += [[v1+j, v1+jn, v2+j], [v1+jn, v2+jn, v2+j]]
        return trimesh.Trimesh(vertices=np.array(all_v),
                               faces=np.array(all_f), process=False)

    endo_loft = loft(rings_endo)
    epi_loft  = loft(rings_epi)
    if len(endo_loft.vertices) == 0: return trimesh.Trimesh()
    if len(epi_loft.vertices)  == 0: return endo_loft
    return trimesh.util.concatenate([endo_loft, epi_loft])


# ══════════════════════════════════════════════════════════════
# Generate
# ══════════════════════════════════════════════════════════════

print('\n--- Classical methods ---')

print('1. Convex Hull...')
mesh_convex = safe_mesh(mesh_via_convex_hull, dense_xyz_n)
print(f'   {len(mesh_convex.vertices):,}v | {len(mesh_convex.faces):,}f')

print('2. Alpha Shape (α=0.30)...')
mesh_alpha = safe_mesh(mesh_via_alpha_shape, dense_xyz_n, 0.30)
print(f'   {len(mesh_alpha.vertices):,}v | {len(mesh_alpha.faces):,}f')

print('3. Gaussian Implicit...')
mesh_gaussian = safe_mesh(mesh_via_gaussian, dense_xyz_n)
print(f'   {len(mesh_gaussian.vertices):,}v | {len(mesh_gaussian.faces):,}f')

print('4. Slice Loft...')
mesh_loft = safe_mesh(mesh_via_slice_loft, lv_slices)
print(f'   {len(mesh_loft.vertices):,}v | {len(mesh_loft.faces):,}f')

print('✅ Done')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Method Comparison — 2×3 grid
# ══════════════════════════════════════════════════════════════

all_meshes = [
    # Neural
    ('INR — Endo only',     endo_mesh,             '#27ae60', 0.90, 'Neural'),
    ('INR — Endo + Epi',    [epi_mesh, endo_mesh],
                            ['#e74c3c', '#27ae60'], [0.30, 0.85], 'Neural'),
    # Classical
    ('Convex Hull\n(baseline)',      mesh_convex,   '#3498db', 0.80, 'Classical'),
    ('Alpha Shape\n(α=0.30)',        mesh_alpha,    '#9b59b6', 0.80, 'Classical'),
    ('Gaussian Implicit\n(KDE)',     mesh_gaussian, '#e67e22', 0.80, 'Classical'),
    ('Slice Loft\n(clinical std.)',  mesh_loft,     '#95a5a6', 0.80, 'Classical'),
]

# Global bounds
all_verts_list = []
for info in all_meshes:
    ms = info[1] if isinstance(info[1], list) else [info[1]]
    for m in ms:
        if len(m.vertices) > 0:
            all_verts_list.append(m.vertices)
all_verts_combined = np.vstack(all_verts_list) if all_verts_list else dense_xyz_n

plt.style.use('dark_background')
fig = plt.figure(figsize=(30, 20), facecolor='#0d0d0d')
fig.suptitle(
    'ACDC LV 3D Reconstruction — INR vs. Classical Methods',
    color='white', fontsize=16, fontweight='bold', y=0.99)

category_color = {'Neural': '#f1c40f', 'Classical': '#95a5a6'}

for idx, info in enumerate(all_meshes):
    name, mesh_or_list, color, alpha, category = info
    ax = fig.add_subplot(2, 3, idx+1, projection='3d', facecolor='#0d0d0d')

    # Category badge in title
    cat_col = category_color[category]
    ax.set_title(f'[{category}]\n{name}',
                 color=cat_col, fontsize=11, fontweight='bold', pad=6)
    ax.set_axis_off()
    ax.view_init(elev=20, azim=45)

    if isinstance(mesh_or_list, list):
        for m, c, a in zip(mesh_or_list, color, alpha):
            draw_mesh(ax, m, c, a, max_faces=5000)
    else:
        draw_mesh(ax, mesh_or_list, color, alpha, max_faces=5000)

    # Input contour rings (subtle)
    ax.scatter(*xyz_n[tissue==0].T, c='#7fbf7f', s=3, alpha=0.4, depthshade=True)
    ax.scatter(*xyz_n[tissue==1].T, c='#bf7f7f', s=3, alpha=0.4, depthshade=True)

    _lim(ax, all_verts_combined)

plt.tight_layout()
plt.savefig('/kaggle/working/acdc_method_comparison.png',
            dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
plt.style.use('default')
print('✅ Saved → /kaggle/working/acdc_method_comparison.png')

# ── Stats table ───────────────────────────────────────────────
print('\n' + '='*75)
print(f'{"Method":<30} | {"Category":<10} | {"Verts":>8} | {"Faces":>8} | {"Watertight":>10}')
print('-'*75)

stats = [
    ('INR Endo',            'Neural',    endo_mesh),
    ('INR Endo+Epi',        'Neural',    trimesh.util.concatenate([endo_mesh, epi_mesh])
                                         if len(epi_mesh.vertices) > 0 else endo_mesh),
    ('Convex Hull',         'Classical', mesh_convex),
    ('Alpha Shape (α=0.30)','Classical', mesh_alpha),
    ('Gaussian Implicit',   'Classical', mesh_gaussian),
    ('Slice Loft',          'Classical', mesh_loft),
]

for name, cat, m in stats:
    if len(m.vertices) > 0:
        wt = '✓' if m.is_watertight else '✗'
        print(f'{name:<30} | {cat:<10} | {len(m.vertices):>8,} | {len(m.faces):>8,} | {wt:>10}')
    else:
        print(f'{name:<30} | {cat:<10} | {"(empty)":>8} | {"":>8} | {"":>10}')

print('='*75)

## AHA 17-Segment Wall Thickness Analysis

Lightweight Matplotlib-based AHA analysis ported from the ES notebook.
Works for **both ED and ES** frames — just point to the correct NIfTI and checkpoint.

### Visualisations:
1. **AHA segment assignment** on the endo mesh (static 3D)
2. **Bull's eye plot** — 17-segment wall thickness polar map
3. **Cross-section slicer** — axial WT-coloured slices through the LV
4. **Radar chart** — per-segment mean WT

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as mcm
from scipy.spatial import cKDTree
from collections import defaultdict

# ══════════════════════════════════════════════════════════════
# AHA 17-Segment Helper Functions
# ══════════════════════════════════════════════════════════════

def find_boundary_verts(faces):
    """Return indices of boundary (open rim) vertices."""
    edge_count = defaultdict(int)
    for f in faces:
        for i in range(3):
            e = tuple(sorted([f[i], f[(i+1) % 3]]))
            edge_count[e] += 1
    return np.array(sorted({v for (a,b), c in edge_count.items()
                            if c == 1 for v in (a,b)}), dtype=np.int32)


def find_apex_base(verts, faces):
    """Find apex vertex and base ring from mesh topology."""
    bv = find_boundary_verts(faces)
    if len(bv) == 0:
        # Closed mesh: use z extremes
        base_center = verts[verts[:,2].argmax()]
        apex_idx = int(np.argmin(np.linalg.norm(verts - base_center, axis=1)))
        return apex_idx, np.array([], dtype=np.int32), base_center
    base_center = verts[bv].mean(axis=0)
    apex_idx = int(np.argmax(np.linalg.norm(verts - base_center, axis=1)))
    return apex_idx, bv, base_center


def assign_aha_segments(verts, apex_idx, base_center, anterior_ref=None):
    """Assign each vertex to one of the 17 AHA segments.

    Basal (1-6): 6×60° sectors, top 34%
    Mid (7-12): 6×60° sectors, mid 33%
    Apical (13-16): 4×90° sectors, next 23%
    Apex (17): bottom 10%
    """
    apex = verts[apex_idx]
    long_vec = base_center - apex
    long_len = np.linalg.norm(long_vec) + 1e-8
    long_unit = long_vec / long_len

    proj = np.clip(np.dot(verts - apex, long_unit) / long_len, 0, 1)
    radial = verts - (apex + proj[:, None] * long_vec)

    if anterior_ref is None:
        mid_mask = (proj > 0.33) & (proj < 0.67)
        if mid_mask.sum() >= 3:
            _, _, vt = np.linalg.svd(radial[mid_mask], full_matrices=False)
            anterior_ref = vt[0]
        else:
            anterior_ref = np.array([1.0, 0.0, 0.0])

    lateral_ref = np.cross(long_unit, anterior_ref)
    lateral_ref /= np.linalg.norm(lateral_ref) + 1e-8

    theta_deg = (np.degrees(
        np.arctan2(radial @ lateral_ref, radial @ anterior_ref)) + 360) % 360

    seg = np.zeros(len(verts), dtype=np.int32)
    seg[proj <= 0.10]                            = 17
    mask_ap = (proj > 0.10) & (proj <= 0.33);    seg[mask_ap] = 13 + (theta_deg[mask_ap] // 90).astype(int) % 4
    mask_md = (proj > 0.33) & (proj <= 0.67);    seg[mask_md] =  7 + (theta_deg[mask_md] // 60).astype(int) % 6
    mask_bs = proj > 0.67;                        seg[mask_bs] =  1 + (theta_deg[mask_bs] // 60).astype(int) % 6
    return seg


def compute_aha_wall_thickness(endo_verts, epi_verts, faces):
    """Compute per-AHA-segment mean wall thickness via KD-tree."""
    apex_idx, _, base_center = find_apex_base(endo_verts, faces)
    tree = cKDTree(epi_verts)
    wt, _ = tree.query(endo_verts)
    seg = assign_aha_segments(endo_verts, apex_idx, base_center)
    wt_map = {}
    for s in range(1, 18):
        m = seg == s
        wt_map[s] = float(wt[m].mean()) if m.any() else float('nan')
    return wt_map, wt, seg


print('✅ AHA functions defined')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Bull's Eye Plot (AHA 17-segment) — lightweight Matplotlib
# ══════════════════════════════════════════════════════════════

_SEG_ANGLE = {
    1: 90, 2: 30, 3: -30, 4: -90, 5: -150, 6: 150,   # basal
    7: 90, 8: 30, 9: -30, 10: -90, 11: -150, 12: 150,  # mid
    13: 90, 14: 0, 15: -90, 16: 180,                    # apical
    17: 0,                                               # apex
}
_SEG_WIDTH = {**{s: 60 for s in range(1,13)},
              **{s: 90 for s in range(13,17)}, 17: 360}
_RING_R    = {**{s: (0.67, 1.00) for s in range(1,7)},
              **{s: (0.34, 0.67) for s in range(7,13)},
              **{s: (0.10, 0.34) for s in range(13,17)},
              17: (0.00, 0.10)}
_RING_NAME = {**{s: 'Basal'  for s in range(1,7)},
              **{s: 'Mid'    for s in range(7,13)},
              **{s: 'Apical' for s in range(13,17)},
              17: 'Apex'}
_AHA_NAMES = {
    1:'Basal anterior',       2:'Basal anteroseptal',
    3:'Basal inferoseptal',   4:'Basal inferior',
    5:'Basal inferolateral',  6:'Basal anterolateral',
    7:'Mid anterior',         8:'Mid anteroseptal',
    9:'Mid inferoseptal',    10:'Mid inferior',
    11:'Mid inferolateral',  12:'Mid anterolateral',
    13:'Apical anterior',    14:'Apical septal',
    15:'Apical inferior',    16:'Apical lateral',
    17:'Apex',
}


def bullseye_plot(wt_dict, patient_id='Patient', phase='ED',
                  vmin=3.0, vmax=14.0, cmap='RdYlGn'):
    """Draw AHA 17-segment bull's eye wall thickness plot.

    Parameters
    ----------
    wt_dict : dict {seg_id 1-17: WT in mm}
    """
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    cmap_fn = plt.get_cmap(cmap)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for seg_id in range(1, 18):
        val   = wt_dict.get(seg_id, float('nan'))
        color = cmap_fn(norm(val)) if not np.isnan(val) else (.85,.85,.85,1)
        ang_c = _SEG_ANGLE[seg_id]
        ang_w = _SEG_WIDTH[seg_id]
        r_in, r_out = _RING_R[seg_id]

        t1 = ang_c - ang_w / 2 + 90
        t2 = ang_c + ang_w / 2 + 90
        w = mpatches.Wedge((0,0), r_out, t1, t2, width=r_out - r_in,
                           facecolor=color, edgecolor='white', linewidth=0.6)
        ax.add_patch(w)

        if not np.isnan(val):
            ang_rad = np.radians(ang_c + 90)
            r_m = (r_in + r_out) / 2
            x, y = r_m * np.cos(ang_rad), r_m * np.sin(ang_rad)
            ax.text(x, y, f'{seg_id}\n{val:.1f}', ha='center', va='center',
                    fontsize=6.5, fontweight='bold',
                    color='black' if val > (vmin + vmax) / 2 else 'white')

    ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(f'{patient_id} — {phase} Wall Thickness (mm)', fontsize=10, pad=6)

    sm = mcm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02, label='WT (mm)', shrink=0.85)
    plt.tight_layout()
    return fig


# ══════════════════════════════════════════════════════════════
# Cross-section slicer — Matplotlib (lightweight, no Plotly)
# ══════════════════════════════════════════════════════════════

def cross_section_plot(endo_verts, epi_verts, faces, n_slices=6, phase='ED'):
    """Show WT-coloured cross-sections through the LV."""
    apex_idx, _, base_center = find_apex_base(endo_verts, faces)
    apex = endo_verts[apex_idx]
    long_v = base_center - apex
    long_l = np.linalg.norm(long_v) + 1e-8
    long_u = long_v / long_l

    t_endo = np.dot(endo_verts - apex, long_u) / long_l

    tree = cKDTree(epi_verts)
    wt, _ = tree.query(endo_verts)

    # Local coordinate frame
    radial = endo_verts - (apex + t_endo[:, None] * long_v)
    _, _, vt = np.linalg.svd(radial, full_matrices=False)
    e1, e2 = vt[0], vt[1]

    slice_t = np.linspace(0.08, 0.92, n_slices)
    ncols = min(4, n_slices)
    nrows = int(np.ceil(n_slices / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
    axes = np.atleast_2d(axes)
    sc = None

    for si, t_target in enumerate(slice_t):
        row, col = divmod(si, ncols)
        ax = axes[row, col]
        bw = 0.5 / n_slices
        mask = np.abs(t_endo - t_target) < bw

        if mask.sum() < 5:
            ax.set_title(f't={t_target:.2f} (empty)', fontsize=8); ax.axis('off')
            continue

        en2d = np.column_stack([endo_verts[mask] @ e1, endo_verts[mask] @ e2])
        wt_slice = wt[mask]

        sc = ax.scatter(en2d[:,0], en2d[:,1], c=wt_slice, s=3,
                        cmap='YlOrRd', vmin=0, vmax=np.percentile(wt, 98))

        level = 'Apex' if t_target < 0.15 else ('Apical' if t_target < 0.35 else
                ('Mid' if t_target < 0.68 else 'Basal'))
        ax.set_title(f'{level} (t={t_target:.2f})\nWT={wt_slice.mean():.1f}mm', fontsize=8)
        ax.set_aspect('equal'); ax.axis('off')

    for si in range(n_slices, nrows * ncols):
        row, col = divmod(si, ncols)
        axes[row, col].axis('off')

    plt.suptitle(f'Cross-Section Slices — {phase} WT', fontsize=11, y=1.01)
    plt.tight_layout()
    if sc is not None:
        plt.colorbar(sc, ax=axes.ravel().tolist(), label='WT (mm)',
                     fraction=0.02, pad=0.02)
    return fig


# ══════════════════════════════════════════════════════════════
# Radar chart — per-segment mean WT
# ══════════════════════════════════════════════════════════════

def radar_chart(wt_dict, phase='ED'):
    """Polar radar chart of per-segment wall thickness."""
    fig, ax = plt.subplots(1, 1, figsize=(7, 7), subplot_kw=dict(polar=True))
    means = [wt_dict.get(s, 0) for s in range(1, 18)]
    angles = np.linspace(0, 2 * np.pi, 17, endpoint=False).tolist()
    means_c = means + [means[0]]
    angles_c = angles + [angles[0]]
    labels = [f'S{s}' for s in range(1, 18)]

    ax.fill(angles_c, means_c, alpha=0.25, color='steelblue')
    ax.plot(angles_c, means_c, 'o-', color='steelblue', lw=1.5, ms=4)
    ax.set_xticks(angles)
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_title(f'{phase} Wall Thickness by AHA Segment', fontsize=11, pad=20)
    plt.tight_layout()
    return fig


print('✅ Visualisation functions defined (bull\'s eye, cross-section, radar)')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Compute AHA wall thickness on the predicted meshes
# ══════════════════════════════════════════════════════════════

PHASE = 'ED'  # Change to 'ES' when running on systole frame
PATIENT_ID = 'ACDC patient001'

if len(endo_mesh.vertices) == 0 or len(epi_mesh.vertices) == 0:
    print('⚠ No valid meshes to analyse — check prediction above.')
else:
    # 1. Compute per-segment wall thickness
    wt_dict = compute_aha_wall_thickness(
        endo_mesh.vertices, epi_mesh.vertices, endo_mesh.faces
    )

    print(f'AHA 17-Segment Wall Thickness ({PHASE})')
    print('─' * 44)
    for seg in range(1, 18):
        name = _AHA_NAMES.get(seg, f'Seg {seg}')
        val  = wt_dict.get(seg, float('nan'))
        bar  = '█' * int(val / 0.5) if not np.isnan(val) else ''
        print(f'  S{seg:02d}  {name:<24s} {val:6.2f} mm  {bar}')

    mean_wt = np.nanmean([v for v in wt_dict.values()])
    print(f'\n  Mean WT: {mean_wt:.2f} mm')
    print(f'  Normal range (ED): 6–11 mm  |  (ES): 8–16 mm')

In [ ]:
# ══════════════════════════════════════════════════════════════
# AHA 3D Segment Visualisation (static Matplotlib)
# ══════════════════════════════════════════════════════════════

if len(endo_mesh.vertices) > 0:
    seg_map = assign_aha_segments(
        endo_mesh.vertices,
        *find_apex_base(endo_mesh.vertices, endo_mesh.faces)[::2]  # apex_idx, base_center
    )
    seg_arr = np.array([seg_map[i] for i in range(len(endo_mesh.vertices))])

    cmap17 = plt.get_cmap('tab20', 17)
    face_colors = cmap17(seg_arr[endo_mesh.faces].mean(axis=1).astype(int) / 17)

    fig = plt.figure(figsize=(10, 7), facecolor='#0d0d0d')
    ax  = fig.add_subplot(111, projection='3d', facecolor='#0d0d0d')
    v = endo_mesh.vertices
    ax.add_collection3d(Poly3DCollection(
        v[endo_mesh.faces], facecolors=face_colors, edgecolor='none', alpha=0.85))
    pad = 0.08
    for fn, d in zip([ax.set_xlim, ax.set_ylim, ax.set_zlim], range(3)):
        fn(v[:,d].min() - pad, v[:,d].max() + pad)
    ax.set_title(f'{PATIENT_ID} — AHA Segments ({PHASE})', color='white', fontsize=11)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.xaxis.label.set_color('gray'); ax.yaxis.label.set_color('gray'); ax.zaxis.label.set_color('gray')
    ax.tick_params(colors='gray')

    # Legend
    patches = [mpatches.Patch(color=cmap17(s / 17), label=f'S{s+1} {_AHA_NAMES[s+1]}')
               for s in range(17)]
    ax.legend(handles=patches, loc='upper left', fontsize=5.5,
              ncol=2, framealpha=0.6, facecolor='#1a1a1a', labelcolor='white')
    plt.tight_layout()
    plt.show()
else:
    print('⚠ Skip: no endo mesh')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Bull's eye + Radar chart side by side
# ══════════════════════════════════════════════════════════════

if len(endo_mesh.vertices) > 0:
    fig_be  = bullseye_plot(wt_dict, patient_id=PATIENT_ID, phase=PHASE,
                            vmin=3.0 if PHASE == 'ES' else 2.0,
                            vmax=14.0 if PHASE == 'ES' else 12.0)
    plt.show()

    fig_rad = radar_chart(wt_dict, phase=PHASE)
    plt.show()
else:
    print('⚠ Skip: no valid meshes')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cross-section slicer
# ══════════════════════════════════════════════════════════════

if len(endo_mesh.vertices) > 0 and len(epi_mesh.vertices) > 0:
    fig_cs = cross_section_plot(
        endo_mesh.vertices, epi_mesh.vertices, endo_mesh.faces,
        n_slices=8, phase=PHASE
    )
    plt.show()
else:
    print('⚠ Skip: meshes missing')

---

# Combined Model Inference (ED + ES)

## Phase-Aware INR from `training-model-v2`

The **combined model** was trained on both ED and ES data with a 5th input feature:
`[x, y, z, tissue, phase]` where `phase=0.0` for ED and `phase=1.0` for ES.

This section:
1. Rebuilds the model architecture with `input_dim=5`
2. Loads the combined checkpoint (`inr_combined.pt`)
3. Finds **both ED and ES** frames for the ACDC patient
4. Runs inference on each phase separately
5. Produces the same visualisations as the ES dataset notebook:
   - **Interactive 3D** (Plotly): endo+epi overlay, AHA segment map, WT heatmap
   - **Cross-section slicer** with WT colouring
   - **Bull's eye plots** for ED and ES side-by-side
   - **Radar chart** comparison
   - **ED vs ES wall thickness change** analysis

In [ ]:
%pip install -q plotly

## Combined Model Architecture & Loading

The `PointNetEncoder` now accepts `input_dim=5` to include the phase feature.
Everything else (decoder, Fourier PE) is identical to the original.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Combined model architecture (input_dim=5: xyz + tissue + phase)
# ══════════════════════════════════════════════════════════════

class PointNetEncoderV2(nn.Module):
    """Phase-aware PointNet encoder with input_dim parameter."""
    def __init__(self, input_dim=5, latent_dim=256):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Linear(input_dim, 64),  nn.LayerNorm(64),  nn.ReLU())
        self.conv2 = nn.Sequential(nn.Linear(64, 128),        nn.LayerNorm(128), nn.ReLU())
        self.conv3 = nn.Sequential(nn.Linear(128, 256),       nn.LayerNorm(256), nn.ReLU())
        self.tissue_proj = nn.Sequential(nn.Linear(256, 128), nn.ReLU())
        self.agg = nn.Sequential(
            nn.Linear(512, 512), nn.LayerNorm(512), nn.ReLU(),
            nn.Linear(512, latent_dim))

    def forward(self, x, mask):
        B, N, _ = x.shape
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        neg_inf  = torch.full_like(h, float('-inf'))
        h_masked = torch.where(mask.unsqueeze(-1), h, neg_inf)
        g_all    = h_masked.max(dim=1).values
        tissue   = x[:, :, 3]
        hp       = self.tissue_proj(h)

        def tissue_pool(label):
            t_mask = (tissue == label) & mask
            t_neg  = torch.full_like(hp, float('-inf'))
            t_h    = torch.where(t_mask.unsqueeze(-1), hp, t_neg)
            pooled = t_h.max(dim=1).values
            has    = t_mask.any(dim=1, keepdim=True).float()
            return pooled * has

        g_endo = tissue_pool(0.0)
        g_epi  = tissue_pool(1.0)
        return self.agg(torch.cat([g_all, g_endo, g_epi], dim=-1))


class OccupancyNetworkV2(nn.Module):
    """Combined model with configurable input_dim for phase-aware encoding."""
    def __init__(self, cfg):
        super().__init__()
        self.encoder = PointNetEncoderV2(
            input_dim=cfg.get('input_dim', 5),
            latent_dim=cfg['latent_dim'])
        self.fourier = FourierPE(L=cfg['fourier_L'])
        self.decoder = INRDecoder(
            latent_dim=cfg['latent_dim'], fourier_L=cfg['fourier_L'],
            hidden=cfg['decoder_hidden'], n_layers=cfg['decoder_layers'],
            skip_layer=cfg['skip_layer'])

    def encode(self, contour, mask):
        return self.encoder(contour, mask)

    def decode(self, z, query_xyz):
        pe = self.fourier(query_xyz)
        return self.decoder(z, pe)

    def forward(self, contour, mask, query_xyz):
        z = self.encode(contour, mask)
        endo_logit, epi_logit = self.decode(z, query_xyz)
        return endo_logit, epi_logit, z


# ── Config for combined model ─────────────────────────────────
CFG_COMBINED = dict(
    input_dim      = 5,       # xyz + tissue + phase
    latent_dim     = 256,
    fourier_L      = 6,
    decoder_hidden = 512,
    decoder_layers = 8,
    skip_layer     = 4,
    grid_res       = 64,
    iso_thresh     = 0.5,
)

# ── Combined checkpoint path ─────────────────────────────────
# UPDATE THIS to your combined model checkpoint:
CKPT_COMBINED = '/kaggle/input/models/andrefce/inr-final/pytorch/combined/1/inr_combined_final.pt'

combined_model = OccupancyNetworkV2(CFG_COMBINED).to(DEVICE)
print(f'Combined model parameters: {sum(p.numel() for p in combined_model.parameters() if p.requires_grad):,}')

# ── Load checkpoint ───────────────────────────────────────────
ckpt_c = torch.load(CKPT_COMBINED, map_location=DEVICE, weights_only=False)

# Handle DataParallel prefix in state_dict keys
state_dict = ckpt_c['model_state']
if any(k.startswith('module.') for k in state_dict):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

combined_model.load_state_dict(state_dict)
combined_model.eval()
print(f'✅ Combined checkpoint loaded')
print(f'   Epoch    : {ckpt_c.get("epoch", "?")}')
print(f'   Val loss : {ckpt_c.get("val_loss", float("nan")):.4f}')
if 'cfg' in ckpt_c:
    print(f'   Mode     : {ckpt_c["cfg"].get("mode", "?")}')
    print(f'   Input dim: {ckpt_c["cfg"].get("input_dim", "?")}')

## Find ED & ES Frames for ACDC Patient

Auto-detects ED/ES from `Info.cfg` or falls back to LV voxel volume
(max volume = ED, min volume = ES).

In [ ]:
# ══════════════════════════════════════════════════════════════
# Find ED and ES ground-truth NIfTI files for an ACDC patient
# ══════════════════════════════════════════════════════════════
import re, configparser

ACDC_PATIENT_DIR = Path('/kaggle/input/datasets/andrefce/realmri/training/patient001')

def resolve_nii_path(path):
    """Handle nested-folder NIfTI layout (dir named *.nii containing real .nii)."""
    p = Path(path)
    if p.is_file():
        return p
    if p.is_dir():
        for pat in ('*.nii.gz', '*.nii'):
            hits = sorted(p.glob(pat))
            if hits:
                return hits[0]
        raise FileNotFoundError(f"No NIfTI inside directory: {p}")
    raise FileNotFoundError(f"Path does not exist: {p}")


def find_ed_es_frames(patient_dir):
    """
    Return (ed_gt_path, es_gt_path) for an ACDC patient directory.
    Tries Info.cfg first, then falls back to LV volume heuristic.
    """
    patient_dir = Path(patient_dir)

    # Find all GT entries
    gt_entries = {}
    for child in sorted(patient_dir.iterdir()):
        m = re.search(r'_frame(\d+)_gt', child.name)
        if m and (child.suffix in ('.gz', '.nii') or child.is_dir()):
            gt_entries[int(m.group(1))] = child

    if len(gt_entries) < 2:
        raise ValueError(f'Need ≥2 GT frames, found {len(gt_entries)} in {patient_dir}')

    # Try Info.cfg
    ed_frame = es_frame = None
    cfg_path = patient_dir / 'Info.cfg'
    if cfg_path.exists():
        cfg = configparser.ConfigParser()
        with open(cfg_path) as f:
            cfg.read_string('[DEFAULT]\n' + f.read())
        try:
            ed_frame = int(cfg['DEFAULT']['ED'])
            es_frame = int(cfg['DEFAULT']['ES'])
        except (KeyError, ValueError):
            pass

    # Fallback: detect from LV volume
    if ed_frame is None or es_frame is None:
        lv_vols = {}
        for fn, gt_path in gt_entries.items():
            try:
                real = resolve_nii_path(gt_path)
                img  = nib.load(str(real))
                vol  = img.get_fdata(dtype=np.float32)
                if vol.ndim == 4:
                    vol = vol[..., 0]
                lv_vols[fn] = int((vol == LBL_LV).sum())
            except Exception:
                lv_vols[fn] = 0
        ed_frame = max(lv_vols, key=lv_vols.get)
        es_frame = min((k for k, v in lv_vols.items() if v > 0),
                       key=lv_vols.get)

    if ed_frame not in gt_entries or es_frame not in gt_entries:
        raise ValueError(f'ED={ed_frame} or ES={es_frame} not found in GT entries')

    return gt_entries[ed_frame], gt_entries[es_frame], ed_frame, es_frame


ed_gt_path, es_gt_path, ed_frame_no, es_frame_no = find_ed_es_frames(ACDC_PATIENT_DIR)
print(f'Patient: {ACDC_PATIENT_DIR.name}')
print(f'  ED frame {ed_frame_no}: {ed_gt_path.name}')
print(f'  ES frame {es_frame_no}: {es_gt_path.name}')

## Extract Contours for Both Phases

Loads both ED and ES NIfTI segmentations, extracts endo/epi contours,
normalises, and appends the **phase feature** (0=ED, 1=ES).

In [ ]:
# ══════════════════════════════════════════════════════════════
# Extract contours for ED and ES phases
# ══════════════════════════════════════════════════════════════

def load_and_extract_contours(gt_path, n_pts=N_PTS_PER_RING):
    """
    Load a NIfTI GT segmentation and extract endo+epi contour points.
    Returns (xyz_norm, tissue_labels, centroid, scale, lv_slices, data, affine).
    """
    real_path = resolve_nii_path(gt_path)
    nii_  = nib.load(str(real_path))
    data_ = nii_.get_fdata().astype(np.int32)
    if data_.ndim == 4:
        data_ = data_[..., 0]
    aff_  = nii_.affine
    H_, W_, S_ = data_.shape

    dz_ = float(nii_.header.get_zooms()[2])

    slices_ = sorted([s for s in range(S_)
                      if (data_[:,:,s] == LBL_LV).any() or (data_[:,:,s] == LBL_MYO).any()])

    def voxel_to_world_local(rows, cols, slice_idx):
        vox = np.column_stack([cols, rows, np.zeros(len(rows)), np.ones(len(rows))])
        world = (aff_ @ vox.T).T
        world[:, 2] = slice_idx * dz_
        return world[:, :3]

    contour_pts = []
    for s in slices_:
        seg = data_[:, :, s]
        # Endo (LV cavity boundary)
        lv_mask = (seg == LBL_LV).astype(np.uint8)
        if lv_mask.sum() > 10:
            ctrs = find_contours(lv_mask, 0.5)
            if ctrs:
                ring = max(ctrs, key=contour_area)
                ring = sample_contour(ring, n_pts)
                world = voxel_to_world_local(ring[:, 0], ring[:, 1], s)
                contour_pts.append(np.column_stack([world, np.zeros(len(world))]))
        # Epi (LV+Myo outer boundary)
        myo_mask = ((seg == LBL_MYO) | (seg == LBL_LV)).astype(np.uint8)
        if myo_mask.sum() > 10:
            ctrs = find_contours(myo_mask, 0.5)
            if ctrs:
                ring = max(ctrs, key=contour_area)
                ring = sample_contour(ring, n_pts)
                world = voxel_to_world_local(ring[:, 0], ring[:, 1], s)
                contour_pts.append(np.column_stack([world, np.ones(len(world))]))

    raw = np.vstack(contour_pts).astype(np.float32)
    xyz_ = raw[:, :3]
    tissue_ = raw[:, 3]

    centroid_ = xyz_.mean(0)
    xyz_c_ = xyz_ - centroid_
    scale_ = np.linalg.norm(xyz_c_[:, :2], axis=1).mean()
    xyz_n_ = (xyz_c_ / scale_).astype(np.float32)

    if FLIP_Z:
        xyz_n_[:, 2] = -xyz_n_[:, 2]

    return xyz_n_, tissue_, centroid_, scale_, slices_, data_, aff_


# ── Extract contours for both phases ──────────────────────────
ed_xyz_n, ed_tissue, ed_centroid, ed_scale, ed_slices, ed_data, ed_affine = \
    load_and_extract_contours(ed_gt_path)
es_xyz_n, es_tissue, es_centroid, es_scale, es_slices, es_data, es_affine = \
    load_and_extract_contours(es_gt_path)

print(f'ED contours: {len(ed_xyz_n)} pts  (endo={int((ed_tissue==0).sum())}, epi={int((ed_tissue==1).sum())})')
print(f'ES contours: {len(es_xyz_n)} pts  (endo={int((es_tissue==0).sum())}, epi={int((es_tissue==1).sum())})')
print(f'ED scale: {ed_scale:.2f} mm   ES scale: {es_scale:.2f} mm')

## Preview ED & ES Slices Side-by-Side

In [ ]:
# ══════════════════════════════════════════════════════════════
# Preview ED vs ES slices
# ══════════════════════════════════════════════════════════════

common_slices = sorted(set(ed_slices) & set(es_slices))
show_slices = common_slices if common_slices else sorted(set(ed_slices) | set(es_slices))
n_show_c = min(len(show_slices), 8)
show_slices = show_slices[:n_show_c]

fig, axes = plt.subplots(2, n_show_c, figsize=(n_show_c * 2.8, 6.0), facecolor='#0d0d0d')
if n_show_c == 1:
    axes = axes.reshape(2, 1)

for i, s in enumerate(show_slices):
    for row, (d, label) in enumerate([(ed_data, 'ED'), (es_data, 'ES')]):
        ax = axes[row, i]
        if s < d.shape[2]:
            ax.imshow(seg_to_rgb(d[:,:,s]), origin='lower', interpolation='nearest')
        ax.set_title(f'{label} s{s}', color='white', fontsize=7, pad=2)
        ax.axis('off')

for j in range(n_show_c, axes.shape[1]):
    axes[0, j].axis('off')
    axes[1, j].axis('off')

fig.suptitle(f'{ACDC_PATIENT_DIR.name} — ED (frame {ed_frame_no}) vs ES (frame {es_frame_no})',
             color='white', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f'✅ ED has {len(ed_slices)} annotated slices, ES has {len(es_slices)}')

## Phase-Aware Inference (predict_mesh_combined)

The combined model receives `[x, y, z, tissue, phase]` where
`phase=0.0` for ED and `phase=1.0` for ES.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Phase-aware inference function
# ══════════════════════════════════════════════════════════════

def predict_mesh_combined(model_net, contour_xyz, tissue_labels, cfg,
                          phase_val=0.0, grid_res=None, iso_thresh=None,
                          batch_query=65536):
    """
    Run the combined occupancy network with phase feature.

    Parameters
    ----------
    phase_val : float — 0.0 for ED, 1.0 for ES
    """
    if grid_res   is None: grid_res   = cfg['grid_res']
    if iso_thresh is None: iso_thresh = cfg['iso_thresh']

    model_net.eval()

    # Build encoder input: [x, y, z, tissue, phase]
    cont = np.column_stack([
        contour_xyz,
        tissue_labels,
        np.full(len(contour_xyz), phase_val, dtype=np.float32)
    ]).astype(np.float32)

    cont_t = torch.from_numpy(cont).unsqueeze(0).to(DEVICE)
    mask_t = torch.ones(1, len(cont), dtype=torch.bool, device=DEVICE)

    with torch.no_grad():
        z = model_net.encode(cont_t, mask_t)

    # Dense query grid
    pad = 0.15
    lo  = contour_xyz.min(0) - pad
    hi  = contour_xyz.max(0) + pad
    ext = hi - lo
    vox = ext.max() / (grid_res - 1)
    nx, ny, nz = [max(4, int(np.ceil(e / vox)) + 1) for e in ext]
    xs = np.linspace(lo[0], hi[0], nx)
    ys = np.linspace(lo[1], hi[1], ny)
    zs = np.linspace(lo[2], hi[2], nz)
    gx, gy, gz = np.meshgrid(xs, ys, zs, indexing='ij')
    grid_pts = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=-1).astype(np.float32)

    phase_name = 'ED' if phase_val == 0.0 else 'ES'
    print(f'  [{phase_name}] Grid: {nx}×{ny}×{nz}  ({len(grid_pts):,} pts)  voxel≈{vox:.4f}')

    endo_logits = np.zeros(len(grid_pts), np.float32)
    epi_logits  = np.zeros(len(grid_pts), np.float32)

    for start in range(0, len(grid_pts), batch_query):
        pts = torch.from_numpy(grid_pts[start:start+batch_query]).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            el, pl = model_net.decode(z, pts)
        endo_logits[start:start+batch_query] = el[0].cpu().numpy()
        epi_logits[start:start+batch_query]  = pl[0].cpu().numpy()

    occ_endo = torch.sigmoid(torch.from_numpy(endo_logits)).numpy().reshape(nx, ny, nz)
    occ_epi  = torch.sigmoid(torch.from_numpy(epi_logits)).numpy().reshape(nx, ny, nz)

    voxel_size = (hi - lo) / (np.array([nx, ny, nz]) - 1)

    def vol_to_mesh(occ):
        if occ.max() < iso_thresh:
            print(f'  ⚠ [{phase_name}] No surface above threshold')
            return trimesh.Trimesh()
        try:
            verts, faces, _, _ = marching_cubes(occ, level=iso_thresh, spacing=voxel_size)
            verts = verts + lo
            mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=True)
            # Keep only largest connected component
            components = mesh.split(only_watertight=False)
            if len(components) > 1:
                mesh = max(components, key=lambda m: len(m.faces))
            return mesh
        except Exception as e:
            print(f'  Marching cubes failed: {e}')
            return trimesh.Trimesh()

    return vol_to_mesh(occ_endo), vol_to_mesh(occ_epi), occ_endo, occ_epi


print('✅ predict_mesh_combined ready')

## Run Combined Inference — ED and ES

In [ ]:
# ══════════════════════════════════════════════════════════════
# Run combined model on both ED and ES
# ══════════════════════════════════════════════════════════════

PATIENT_ID_C = ACDC_PATIENT_DIR.name

print(f'Running combined model on {PATIENT_ID_C}…\n')

# ── ED inference (phase=0.0) ──────────────────────────────────
print('── End-Diastole (ED) ──')
ed_endo_c, ed_epi_c, ed_occ_endo_c, ed_occ_epi_c = predict_mesh_combined(
    combined_model, ed_xyz_n, ed_tissue, CFG_COMBINED, phase_val=0.0)
print(f'  Endo: {len(ed_endo_c.vertices):,}v / {len(ed_endo_c.faces):,}f')
print(f'  Epi:  {len(ed_epi_c.vertices):,}v / {len(ed_epi_c.faces):,}f')

# ── ES inference (phase=1.0) ──────────────────────────────────
print('\n── End-Systole (ES) ──')
es_endo_c, es_epi_c, es_occ_endo_c, es_occ_epi_c = predict_mesh_combined(
    combined_model, es_xyz_n, es_tissue, CFG_COMBINED, phase_val=1.0)
print(f'  Endo: {len(es_endo_c.vertices):,}v / {len(es_endo_c.faces):,}f')
print(f'  Epi:  {len(es_epi_c.vertices):,}v / {len(es_epi_c.faces):,}f')

print(f'\n✅ Both phases predicted with combined model')

## Interactive 3D Visualisation (Plotly)

Four panels per phase (matching ES_dataset notebook):
1. **Endo + Epi overlay** — semi-transparent epi wrapping the endo surface
2. **AHA 17-segment map** — endo vertices coloured by AHA segment
3. **Wall-thickness heatmap** — endo coloured by per-vertex WT
4. **Epi surface** — for inspection

In [ ]:
import plotly.graph_objects as go

# ── AHA segment colour palette (17 distinct colours) ─────────
_AHA_COLORS = {
    1: '#e6194b', 2: '#3cb44b', 3: '#ffe119', 4: '#4363d8',
    5: '#f58231', 6: '#911eb4', 7: '#42d4f4', 8: '#f032e6',
    9: '#bfef45',10: '#fabed4',11: '#469990',12: '#dcbeff',
   13: '#9A6324',14: '#fffac8',15: '#800000',16: '#aaffc3',
   17: '#808000',
}


def _mesh3d(verts, faces, color_values=None, colorscale=None,
            flat_color=None, opacity=1.0, name='', showscale=False,
            cmin=None, cmax=None, colorbar_title='', hovertext=None):
    """Build a Plotly Mesh3d trace."""
    kw = dict(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        opacity=opacity, name=name, flatshading=False,
        lighting=dict(ambient=0.5, diffuse=0.7, specular=0.3,
                      roughness=0.4, fresnel=0.2),
        lightposition=dict(x=100, y=200, z=300),
    )
    if color_values is not None:
        kw['intensity'] = color_values
        kw['colorscale'] = colorscale or 'YlOrRd'
        kw['showscale'] = showscale
        kw['cmin'] = cmin; kw['cmax'] = cmax
        if showscale:
            kw['colorbar'] = dict(title=colorbar_title, thickness=15, len=0.6)
    elif flat_color:
        kw['color'] = flat_color
    if hovertext is not None:
        kw['hovertext'] = hovertext; kw['hoverinfo'] = 'text'
    return go.Mesh3d(**kw)


def _scene_layout():
    return dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                zaxis=dict(visible=False), aspectmode='data', bgcolor='#fafafa')


def _camera():
    return dict(eye=dict(x=0, y=-2.0, z=0.6),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1))


def visualise_3d_phase(endo_mesh, epi_mesh, phase='ED', patient_id='Patient'):
    """
    Build 4 interactive Plotly figures for one phase.
    Mirrors the ES_dataset notebook's `visualise_3d_sample`.
    """
    endo_v = endo_mesh.vertices.astype(np.float64)
    epi_v  = epi_mesh.vertices.astype(np.float64)
    faces  = endo_mesh.faces.astype(np.int32)

    apex_idx, _, base_center = find_apex_base(endo_v, faces)
    tree = cKDTree(epi_v)
    wt, _ = tree.query(endo_v)
    seg_labels = assign_aha_segments(endo_v, apex_idx, base_center)

    # ── Panel 1: Endo + Epi overlay ──────────────────────────
    fig1 = go.Figure()
    fig1.add_trace(_mesh3d(endo_v, faces, flat_color='steelblue', opacity=1.0, name='Endo'))
    fig1.add_trace(_mesh3d(epi_v, epi_mesh.faces.astype(np.int32),
                           flat_color='coral', opacity=0.25, name='Epi'))
    fig1.update_layout(
        title=dict(text=f'{patient_id} — {phase} Endo + Epi Overlay', font=dict(size=13)),
        scene=_scene_layout(), scene_camera=_camera(),
        margin=dict(l=0, r=0, t=40, b=0),
        legend=dict(x=0.01, y=0.99), width=700, height=550)

    # ── Panel 2: AHA segment map ─────────────────────────────
    hover_seg = [f'S{seg_labels[i]}: {_AHA_NAMES[seg_labels[i]]}<br>WT={wt[i]:.1f}'
                 for i in range(len(endo_v))]
    fig2 = go.Figure()
    fig2.add_trace(_mesh3d(
        endo_v, faces,
        color_values=seg_labels.astype(float),
        colorscale=[[i/16, _AHA_COLORS[i+1]] for i in range(17)],
        cmin=1, cmax=17, showscale=True, colorbar_title='AHA Seg',
        opacity=1.0, name='AHA segments', hovertext=hover_seg))
    fig2.update_layout(
        title=dict(text=f'{patient_id} — {phase} AHA 17-Segment Map', font=dict(size=13)),
        scene=_scene_layout(), scene_camera=_camera(),
        margin=dict(l=0, r=0, t=40, b=0), width=700, height=550)

    # ── Panel 3: Wall-thickness heatmap ──────────────────────
    hover_wt = [f'WT={wt[i]:.2f}<br>S{seg_labels[i]}: {_AHA_NAMES[seg_labels[i]]}'
                for i in range(len(endo_v))]
    fig3 = go.Figure()
    fig3.add_trace(_mesh3d(
        endo_v, faces,
        color_values=wt,
        colorscale='YlOrRd',
        cmin=max(0, np.percentile(wt, 2)), cmax=np.percentile(wt, 98),
        showscale=True, colorbar_title='WT',
        opacity=1.0, name='Wall thickness', hovertext=hover_wt))
    fig3.update_layout(
        title=dict(text=f'{patient_id} — {phase} Wall Thickness Heatmap', font=dict(size=13)),
        scene=_scene_layout(), scene_camera=_camera(),
        margin=dict(l=0, r=0, t=40, b=0), width=700, height=550)

    # ── Panel 4: Epi surface ─────────────────────────────────
    fig4 = go.Figure()
    fig4.add_trace(_mesh3d(epi_v, epi_mesh.faces.astype(np.int32),
                           flat_color='#ff7f50', opacity=0.85, name='Epi surface'))
    fig4.add_trace(_mesh3d(endo_v, faces,
                           flat_color='#4682b4', opacity=0.15, name='Endo (ghost)'))
    fig4.update_layout(
        title=dict(text=f'{patient_id} — {phase} Epi Surface', font=dict(size=13)),
        scene=_scene_layout(), scene_camera=_camera(),
        margin=dict(l=0, r=0, t=40, b=0),
        legend=dict(x=0.01, y=0.99), width=700, height=550)

    for f in [fig1, fig2, fig3, fig4]:
        f.show()

    return fig1, fig2, fig3, fig4


# ── Visualise ED ──────────────────────────────────────────────
if len(ed_endo_c.vertices) > 0 and len(ed_epi_c.vertices) > 0:
    print(f'\n{"═"*60}')
    print(f'  ED — Interactive 3D')
    print(f'{"═"*60}')
    _ = visualise_3d_phase(ed_endo_c, ed_epi_c, phase='ED', patient_id=PATIENT_ID_C)
else:
    print('⚠ ED meshes empty — skip 3D visualisation')

# ── Visualise ES ──────────────────────────────────────────────
if len(es_endo_c.vertices) > 0 and len(es_epi_c.vertices) > 0:
    print(f'\n{"═"*60}')
    print(f'  ES — Interactive 3D')
    print(f'{"═"*60}')
    _ = visualise_3d_phase(es_endo_c, es_epi_c, phase='ES', patient_id=PATIENT_ID_C)
else:
    print('⚠ ES meshes empty — skip 3D visualisation')

## Cross-Section Slicer (ED & ES)

WT-coloured axial cross-sections through the LV + interactive 3D overview
showing slice locations — same style as the ES_dataset notebook.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cross-section slicer with Plotly 3D overview
# ══════════════════════════════════════════════════════════════

def cross_section_with_3d(endo_mesh, epi_mesh, phase='ED', patient_id='Patient',
                          n_slices=8):
    """
    Show axial WT-coloured cross-section rings + 3D overview.
    Matches ES_dataset notebook style.
    """
    endo_v = endo_mesh.vertices.astype(np.float64)
    epi_v  = epi_mesh.vertices.astype(np.float64)
    faces  = endo_mesh.faces.astype(np.int32)

    apex_idx, _, base_center = find_apex_base(endo_v, faces)
    apex   = endo_v[apex_idx]
    long_v = base_center - apex
    long_l = np.linalg.norm(long_v) + 1e-8
    long_u = long_v / long_l

    t_endo = np.dot(endo_v - apex, long_u) / long_l
    t_epi  = np.dot(epi_v  - apex, long_u) / long_l

    tree = cKDTree(epi_v)
    wt, _ = tree.query(endo_v)

    # Local coordinate frame
    radial = endo_v - (apex + t_endo[:, None] * long_v)
    _, _, vt = np.linalg.svd(radial, full_matrices=False)
    e1, e2 = vt[0], vt[1]

    # ── 2D cross-section grid ────────────────────────────────
    slice_t = np.linspace(0.05, 0.95, n_slices)
    ncols   = min(4, n_slices)
    nrows   = int(np.ceil(n_slices / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
    axes = np.atleast_2d(axes)
    sc = None

    for si, t_target in enumerate(slice_t):
        row, col = divmod(si, ncols)
        ax = axes[row, col]
        bw = 0.5 / n_slices
        mask_en = np.abs(t_endo - t_target) < bw
        mask_ep = np.abs(t_epi  - t_target) < bw

        if mask_en.sum() < 5:
            ax.set_title(f't={t_target:.2f}\n(empty)', fontsize=8)
            ax.axis('off'); continue

        en2d = np.column_stack([endo_v[mask_en] @ e1, endo_v[mask_en] @ e2])
        ep2d = np.column_stack([epi_v[mask_ep]  @ e1, epi_v[mask_ep]  @ e2])
        wt_slice = wt[mask_en]

        sc = ax.scatter(en2d[:, 0], en2d[:, 1], c=wt_slice, s=3,
                        cmap='YlOrRd', vmin=0, vmax=np.percentile(wt, 98))
        ax.scatter(ep2d[:, 0], ep2d[:, 1], s=1, c='grey', alpha=0.4, label='epi')

        level = 'Apex' if t_target < 0.15 else ('Apical' if t_target < 0.35 else
                ('Mid' if t_target < 0.68 else 'Basal'))
        ax.set_title(f'{level}  (t={t_target:.2f})\nmean WT={wt_slice.mean():.1f}',
                     fontsize=8)
        ax.set_aspect('equal'); ax.axis('off')

    for si in range(n_slices, nrows * ncols):
        row, col = divmod(si, ncols)
        axes[row, col].axis('off')

    plt.suptitle(f'Cross-section slices — {patient_id} {phase}', fontsize=11, y=1.01)
    plt.tight_layout()
    if sc is not None:
        plt.colorbar(sc, ax=axes.ravel().tolist(), label='WT',
                     fraction=0.02, pad=0.02)
    plt.show()

    # ── 3D overview with coloured slice bands ────────────────
    fig3d = go.Figure()
    fig3d.add_trace(_mesh3d(endo_v, faces, flat_color='steelblue',
                            opacity=0.12, name='Endo'))

    colours = plt.cm.tab10(np.linspace(0, 1, n_slices))
    for si, t_target in enumerate(slice_t):
        bw = 0.5 / n_slices
        mask = np.abs(t_endo - t_target) < bw
        if mask.sum() < 5:
            continue
        c_hex = '#{:02x}{:02x}{:02x}'.format(
            int(colours[si][0]*255), int(colours[si][1]*255), int(colours[si][2]*255))
        level = 'Apex' if t_target < 0.15 else ('Apical' if t_target < 0.35 else
                ('Mid' if t_target < 0.68 else 'Basal'))
        fig3d.add_trace(go.Scatter3d(
            x=endo_v[mask, 0], y=endo_v[mask, 1], z=endo_v[mask, 2],
            mode='markers', marker=dict(size=1.5, color=c_hex),
            name=f'{level} t={t_target:.2f}',
            hovertext=[f'WT={wt[i]:.1f}' for i in np.where(mask)[0]]))
    fig3d.update_layout(
        title=dict(text=f'Slice locations — {patient_id} {phase}', font=dict(size=12)),
        scene=_scene_layout(), scene_camera=_camera(),
        margin=dict(l=0, r=0, t=40, b=0), width=700, height=500)
    fig3d.show()


# ── Run on both phases ────────────────────────────────────────
for endo_m, epi_m, ph in [(ed_endo_c, ed_epi_c, 'ED'), (es_endo_c, es_epi_c, 'ES')]:
    if len(endo_m.vertices) > 0 and len(epi_m.vertices) > 0:
        cross_section_with_3d(endo_m, epi_m, phase=ph, patient_id=PATIENT_ID_C)
    else:
        print(f'⚠ {ph} meshes empty — skip cross-sections')

## AHA Wall Thickness — Combined Model (ED & ES)

Computes per-segment mean WT and prints the table for both phases.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Compute AHA WT for both phases
# ══════════════════════════════════════════════════════════════

wt_results = {}

for endo_m, epi_m, ph, sc in [
    (ed_endo_c, ed_epi_c, 'ED', ed_scale),
    (es_endo_c, es_epi_c, 'ES', es_scale),
]:
    if len(endo_m.vertices) == 0 or len(epi_m.vertices) == 0:
        print(f'⚠ {ph}: meshes empty — cannot compute WT')
        continue

    wt_map, wt_per_v, seg_labels = compute_aha_wall_thickness(
        endo_m.vertices, epi_m.vertices, endo_m.faces)

    # Scale back to mm (meshes are in normalised space)
    wt_map_mm = {s: v * sc for s, v in wt_map.items()}
    wt_results[ph] = wt_map_mm

    print(f'\n{"═"*50}')
    print(f'  AHA 17-Segment Wall Thickness — {ph}  (scale={sc:.1f} mm)')
    print(f'{"═"*50}')
    rings = [('Basal', range(1,7)), ('Mid', range(7,13)),
             ('Apical', range(13,17)), ('Apex', [17])]
    for ring_name, segs in rings:
        vals = [wt_map_mm.get(s, float('nan')) for s in segs]
        mean_wt = np.nanmean(vals)
        seg_str = '  '.join(f'S{s}:{wt_map_mm.get(s, float("nan")):.1f}' for s in segs)
        print(f'  {ring_name:8s}  mean={mean_wt:.1f} mm  |  {seg_str}')

    overall = np.nanmean(list(wt_map_mm.values()))
    print(f'\n  Overall mean WT: {overall:.1f} mm')

if 'ED' in wt_results and 'ES' in wt_results:
    print(f'\n{"─"*50}')
    print(f'  Expected: ED WT 6–11 mm  |  ES WT 8–16 mm (wall thickens at systole)')
    ed_mean = np.nanmean(list(wt_results['ED'].values()))
    es_mean = np.nanmean(list(wt_results['ES'].values()))
    print(f'  Measured: ED={ed_mean:.1f} mm  ES={es_mean:.1f} mm  '
          f'ΔWT={es_mean - ed_mean:.1f} mm')

## Bull's Eye — ED vs ES Side-by-Side

In [ ]:
# ══════════════════════════════════════════════════════════════
# Bull's eye — ED vs ES side by side
# ══════════════════════════════════════════════════════════════

def _draw_bullseye_ax(ax, wt_dict, title, vmin=3.0, vmax=14.0, cmap='RdYlGn'):
    """Draw a bull's eye on a given axis (for side-by-side layout)."""
    cmap_fn = plt.get_cmap(cmap)
    norm    = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for seg_id in range(1, 18):
        val   = wt_dict.get(seg_id, float('nan'))
        color = cmap_fn(norm(val)) if not np.isnan(val) else (.85,.85,.85,1)
        ang_c = _SEG_ANGLE[seg_id]
        ang_w = _SEG_WIDTH[seg_id]
        r_in, r_out = _RING_R[seg_id]

        t1 = ang_c - ang_w / 2 + 90
        t2 = ang_c + ang_w / 2 + 90
        w  = mpatches.Wedge((0,0), r_out, t1, t2, width=r_out - r_in,
                             facecolor=color, edgecolor='white', linewidth=0.6)
        ax.add_patch(w)

        if not np.isnan(val):
            ang_rad = np.radians(ang_c + 90)
            r_m = (r_in + r_out) / 2
            x, y = r_m * np.cos(ang_rad), r_m * np.sin(ang_rad)
            ax.text(x, y, f'{seg_id}\n{val:.1f}', ha='center', va='center',
                    fontsize=6, fontweight='bold',
                    color='black' if val > (vmin + vmax) / 2 else 'white')

    ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=10, pad=6)


if 'ED' in wt_results and 'ES' in wt_results:
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

    # Shared colour range across both phases
    all_vals = [v for d in wt_results.values() for v in d.values() if not np.isnan(v)]
    vmin_c = max(0, np.percentile(all_vals, 2))
    vmax_c = np.percentile(all_vals, 98)

    _draw_bullseye_ax(ax1, wt_results['ED'],
                      f'{PATIENT_ID_C} — ED WT (mm)',
                      vmin=vmin_c, vmax=vmax_c)

    _draw_bullseye_ax(ax2, wt_results['ES'],
                      f'{PATIENT_ID_C} — ES WT (mm)',
                      vmin=vmin_c, vmax=vmax_c)

    # ΔWT bull's eye (ES - ED)
    delta_wt = {s: wt_results['ES'].get(s, float('nan')) - wt_results['ED'].get(s, float('nan'))
                for s in range(1, 18)}
    dvals = [v for v in delta_wt.values() if not np.isnan(v)]
    d_abs = max(abs(min(dvals)), abs(max(dvals))) if dvals else 5
    _draw_bullseye_ax(ax3, delta_wt,
                      f'{PATIENT_ID_C} — ΔWT (ES − ED) (mm)',
                      vmin=-d_abs, vmax=d_abs, cmap='RdBu')

    # Shared colorbar
    sm = mcm.ScalarMappable(cmap='RdYlGn', norm=mcolors.Normalize(vmin=vmin_c, vmax=vmax_c))
    sm.set_array([])
    plt.colorbar(sm, ax=[ax1, ax2], fraction=0.025, pad=0.02, label='WT (mm)', shrink=0.85)

    sm_d = mcm.ScalarMappable(cmap='RdBu', norm=mcolors.Normalize(vmin=-d_abs, vmax=d_abs))
    sm_d.set_array([])
    plt.colorbar(sm_d, ax=ax3, fraction=0.035, pad=0.02, label='ΔWT (mm)', shrink=0.85)

    plt.suptitle(f'{PATIENT_ID_C} — Bull\'s Eye Comparison (Combined Model)',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

elif wt_results:
    # Only one phase available
    ph = list(wt_results.keys())[0]
    fig_be = bullseye_plot(wt_results[ph], patient_id=PATIENT_ID_C, phase=ph)
    plt.show()
else:
    print('⚠ No WT results to plot')

## Radar Chart — ED vs ES Overlay

Compares per-segment wall thickness across cardiac phases on a single
polar radar chart.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Radar chart — ED vs ES overlay
# ══════════════════════════════════════════════════════════════

if len(wt_results) >= 2:
    fig, ax = plt.subplots(1, 1, figsize=(8, 8), subplot_kw=dict(polar=True))

    angles = np.linspace(0, 2 * np.pi, 17, endpoint=False).tolist()
    angles_c = angles + [angles[0]]
    labels = [f'S{s}' for s in range(1, 18)]

    colors_phase = {'ED': 'steelblue', 'ES': 'coral'}

    for ph, color in colors_phase.items():
        if ph not in wt_results:
            continue
        vals = [wt_results[ph].get(s, 0) for s in range(1, 18)]
        vals_c = vals + [vals[0]]
        ax.fill(angles_c, vals_c, alpha=0.15, color=color)
        ax.plot(angles_c, vals_c, 'o-', color=color, lw=1.5, ms=4, label=ph)

    ax.set_xticks(angles)
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_title(f'{PATIENT_ID_C} — WT by AHA Segment (ED vs ES)',
                 fontsize=12, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
    plt.tight_layout()
    plt.show()

elif wt_results:
    ph = list(wt_results.keys())[0]
    fig_r = radar_chart(wt_results[ph], phase=ph)
    plt.show()
else:
    print('⚠ No WT results for radar chart')

## 5-Panel Mesh Visualisation — ED & ES

Input contours → Predicted Endo → Endo + Epi → Mesh + Contours → Top View

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5-panel mesh comparison — one row per phase
# ══════════════════════════════════════════════════════════════

plt.style.use('dark_background')

phase_data = []
for xyz_n_, tissue_, endo_m, epi_m, ph in [
    (ed_xyz_n, ed_tissue, ed_endo_c, ed_epi_c, 'ED'),
    (es_xyz_n, es_tissue, es_endo_c, es_epi_c, 'ES'),
]:
    if len(endo_m.vertices) > 0:
        phase_data.append((xyz_n_, tissue_, endo_m, epi_m, ph))

n_phases = len(phase_data)
if n_phases == 0:
    print('⚠ No valid meshes — skip 5-panel visualisation')
else:
    fig = plt.figure(figsize=(28, 12 * n_phases), facecolor='#0d0d0d')
    fig.suptitle(
        f'{PATIENT_ID_C} — Combined Model 3D LV Reconstruction (ED & ES)\n'
        'INR Occupancy Network (Phase-Aware PointNet + Fourier INR)',
        color='white', fontsize=14, fontweight='bold', y=0.99)

    for row_idx, (xyz_n_, tissue_, endo_m, epi_m, ph) in enumerate(phase_data):
        all_pts_ = np.vstack([
            endo_m.vertices if len(endo_m.vertices) > 0 else np.zeros((1,3)),
            epi_m.vertices  if len(epi_m.vertices)  > 0 else np.zeros((1,3)),
        ])

        panels = [
            (0, f'{ph}: Input SAX Contours',     20, 45),
            (1, f'{ph}: Predicted Endo',         20, 45),
            (2, f'{ph}: Predicted Endo + Epi',   20, 45),
            (3, f'{ph}: Mesh + SAX Contours',    20, 45),
            (4, f'{ph}: Top View (short-axis)',  88,  0),
        ]

        for col_idx, (_, title, elev, azim) in enumerate(panels):
            ax = fig.add_subplot(n_phases, 5, row_idx * 5 + col_idx + 1,
                                 projection='3d', facecolor='#0d0d0d')
            ax.set_title(title, color='white', fontsize=9, fontweight='bold', pad=6)
            ax.set_axis_off()
            ax.view_init(elev=elev, azim=azim)

            if col_idx == 0:
                ax.scatter(*xyz_n_[tissue_==0].T, c='#27ae60', s=10, alpha=0.9, label='Endo')
                ax.scatter(*xyz_n_[tissue_==1].T, c='#e74c3c', s=10, alpha=0.9, label='Epi')
                ax.legend(fontsize=6, facecolor='#1a1a1a', labelcolor='white', loc='upper left')
            elif col_idx == 1:
                draw_mesh(ax, endo_m, '#27ae60', 0.85)
            elif col_idx == 2:
                draw_mesh(ax, epi_m,  '#e74c3c', 0.35)
                draw_mesh(ax, endo_m, '#27ae60', 0.85)
            elif col_idx == 3:
                draw_mesh(ax, epi_m,  '#e74c3c', 0.30)
                draw_mesh(ax, endo_m, '#27ae60', 0.75)
                ax.scatter(*xyz_n_[tissue_==0].T, c='#7fbf7f', s=3, alpha=0.5, depthshade=True)
                ax.scatter(*xyz_n_[tissue_==1].T, c='#bf7f7f', s=3, alpha=0.5, depthshade=True)
            elif col_idx == 4:
                draw_mesh(ax, epi_m,  '#e74c3c', 0.35)
                draw_mesh(ax, endo_m, '#27ae60', 0.85)

            _lim(ax, all_pts_)

    plt.tight_layout()
    plt.show()
    plt.style.use('default')
    print('✅ 5-panel visualisation done')

## Occupancy Field Cross-Sections — ED & ES

Side-by-side endo/epi occupancy fields for both phases.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Occupancy field cross-sections — ED vs ES
# ══════════════════════════════════════════════════════════════

for occ_e, occ_p, ph in [
    (ed_occ_endo_c, ed_occ_epi_c, 'ED'),
    (es_occ_endo_c, es_occ_epi_c, 'ES'),
]:
    nx_, ny_, nz_ = occ_e.shape
    mid_x, mid_y, mid_z = nx_ // 2, ny_ // 2, nz_ // 2

    fig, axes = plt.subplots(2, 3, figsize=(14, 8), facecolor='#0d0d0d')
    titles = ['Axial (z=mid)', 'Coronal (y=mid)', 'Sagittal (x=mid)']
    slices_e = [occ_e[:,:,mid_z], occ_e[:,mid_y,:], occ_e[mid_x,:,:]]
    slices_p = [occ_p[:,:,mid_z], occ_p[:,mid_y,:], occ_p[mid_x,:,:]]

    for col, (title, se, sp) in enumerate(zip(titles, slices_e, slices_p)):
        for row, (occ_sl, label, cmap_) in enumerate([(se, 'Endo', 'Greens'),
                                                       (sp, 'Epi',  'Reds')]):
            ax = axes[row, col]
            ax.imshow(occ_sl.T, origin='lower', cmap=cmap_, vmin=0, vmax=1)
            ax.contour(occ_sl.T, levels=[0.5], colors='white', linewidths=1.2)
            ax.set_title(f'{ph} {label} — {title}', color='white', fontsize=9)
            ax.axis('off')

    fig.suptitle(f'{ph} Occupancy Field Cross-Sections (0.5 threshold)',
                 color='white', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

plt.style.use('default')
print('✅ Occupancy cross-sections shown for both phases')

## AHA 3D Segment Visualisation — ED & ES

Static Matplotlib 3D view of each endo mesh coloured by AHA segment.

In [ ]:
# ══════════════════════════════════════════════════════════════
# AHA 3D segment visualisation — ED & ES side by side
# ══════════════════════════════════════════════════════════════

meshes_to_show = []
for endo_m, ph in [(ed_endo_c, 'ED'), (es_endo_c, 'ES')]:
    if len(endo_m.vertices) > 0:
        meshes_to_show.append((endo_m, ph))

if meshes_to_show:
    fig = plt.figure(figsize=(10 * len(meshes_to_show), 7), facecolor='#0d0d0d')

    for idx, (endo_m, ph) in enumerate(meshes_to_show):
        apex_idx_, _, base_center_ = find_apex_base(endo_m.vertices, endo_m.faces)
        seg_arr = assign_aha_segments(endo_m.vertices, apex_idx_, base_center_)

        cmap17 = plt.get_cmap('tab20', 17)
        face_seg = seg_arr[endo_m.faces].mean(axis=1).astype(int)
        face_colors = cmap17(face_seg / 17)

        ax = fig.add_subplot(1, len(meshes_to_show), idx + 1,
                             projection='3d', facecolor='#0d0d0d')
        v = endo_m.vertices
        ax.add_collection3d(Poly3DCollection(
            v[endo_m.faces], facecolors=face_colors, edgecolor='none', alpha=0.85))
        pad_ = 0.08
        for fn, d in zip([ax.set_xlim, ax.set_ylim, ax.set_zlim], range(3)):
            fn(v[:,d].min() - pad_, v[:,d].max() + pad_)
        ax.set_title(f'{PATIENT_ID_C} — {ph} AHA Segments', color='white', fontsize=11)
        ax.set_axis_off()
        ax.view_init(elev=20, azim=45)

        if idx == 0:
            patches_ = [mpatches.Patch(color=cmap17(s / 17),
                                       label=f'S{s+1} {_AHA_NAMES[s+1]}')
                        for s in range(17)]
            ax.legend(handles=patches_, loc='upper left', fontsize=5,
                      ncol=2, framealpha=0.6, facecolor='#1a1a1a', labelcolor='white')

    plt.tight_layout()
    plt.show()
else:
    print('⚠ No meshes available for AHA 3D visualisation')

## Per-Segment ΔWT Analysis (ES − ED)

Wall thickening per AHA segment — a key functional metric for
identifying regional dysfunction. Healthy myocardium should thicken
by 3–8 mm from ED to ES.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Per-segment wall thickening analysis (ΔWT = ES − ED)
# ══════════════════════════════════════════════════════════════

if 'ED' in wt_results and 'ES' in wt_results:
    print(f'{"═"*60}')
    print(f'  Per-Segment Wall Thickening (ΔWT = ES − ED)')
    print(f'{"═"*60}')
    print(f'  {"Seg":<4} {"Name":<24s} {"ED":>7} {"ES":>7} {"ΔWT":>7} {"Status"}')
    print(f'  {"─"*4} {"─"*24} {"─"*7} {"─"*7} {"─"*7} {"─"*10}')

    delta_vals = []
    for s in range(1, 18):
        ed_v = wt_results['ED'].get(s, float('nan'))
        es_v = wt_results['ES'].get(s, float('nan'))
        dv   = es_v - ed_v if not (np.isnan(ed_v) or np.isnan(es_v)) else float('nan')
        name = _AHA_NAMES.get(s, f'Seg {s}')

        if np.isnan(dv):
            status = '—'
        elif dv < 1.0:
            status = '⚠ hypokinetic'
        elif dv < 3.0:
            status = '↗ mild'
        elif dv <= 8.0:
            status = '✓ normal'
        else:
            status = '↑ hyperkinetic'

        print(f'  S{s:<2d}  {name:<24s} {ed_v:6.1f}  {es_v:6.1f}  {dv:+6.1f}  {status}')
        if not np.isnan(dv):
            delta_vals.append(dv)

    if delta_vals:
        print(f'\n  Mean ΔWT: {np.mean(delta_vals):+.1f} mm')
        print(f'  Expected healthy ΔWT: +3 to +8 mm')

    # ── Bar chart of ΔWT per segment ─────────────────────────
    fig, ax = plt.subplots(1, 1, figsize=(14, 5))

    segs = list(range(1, 18))
    d_vals = [wt_results['ES'].get(s, 0) - wt_results['ED'].get(s, 0) for s in segs]
    colors_bar = ['#e74c3c' if v < 1 else ('#f39c12' if v < 3 else '#27ae60') for v in d_vals]

    bars = ax.bar([f'S{s}' for s in segs], d_vals, color=colors_bar, edgecolor='white', linewidth=0.5)

    # Reference bands
    ax.axhspan(3, 8, alpha=0.1, color='green', label='Normal range (3–8 mm)')
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')

    ax.set_ylabel('ΔWT (ES − ED) [mm]', fontsize=10)
    ax.set_xlabel('AHA Segment', fontsize=10)
    ax.set_title(f'{PATIENT_ID_C} — Wall Thickening per AHA Segment', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Ring separators
    for x_pos in [5.5, 11.5, 15.5]:
        ax.axvline(x_pos, color='grey', linewidth=0.5, linestyle=':')
    for x_pos, lbl in [(2.5, 'Basal'), (8.5, 'Mid'), (13.5, 'Apical'), (16, 'Apex')]:
        ax.text(x_pos, ax.get_ylim()[1] * 0.95, lbl, ha='center', fontsize=8,
                color='grey', fontstyle='italic')

    plt.tight_layout()
    plt.show()
    print('✅ ΔWT analysis complete')

else:
    print('⚠ Need both ED and ES results for ΔWT analysis')

## Summary — Combined Model Inference

| Visualisation | ED | ES | Comparison |
|---|---|---|---|
| Interactive 3D (Plotly) | Endo+Epi, AHA map, WT heatmap, Epi | Same 4 panels | — |
| Cross-section slicer | WT-coloured + 3D overview | Same | — |
| Bull's eye | 17-segment WT | 17-segment WT | ΔWT (ES−ED) |
| Radar chart | — | — | ED vs ES overlay |
| 5-panel mesh | Contours → Endo → Endo+Epi → Overlay → Top | Same row | Side-by-side |
| ΔWT bar chart | — | — | Per-segment thickening |

### To run on a different patient:
```python
ACDC_PATIENT_DIR = Path('/kaggle/input/.../patient042')
```
Then re-run from the **"Find ED & ES Frames"** cell onwards.

### To use a different checkpoint:
```python
CKPT_COMBINED = '/path/to/inr_combined_final.pt'
```